# Transfer functions across the spectrum

The impulse response of the EEG to the broadband gammatone envelope, band by band. Where the
coherence figures ask *how much* of the envelope is tracked at each frequency, this asks *when*:
the same estimator inverse-transformed to lag.

The fit on disk is the parts, not the quotient --
$H(f) = S_{xy}(f) / (S_{xx}(f) + \lambda)$ with $S_{xy}$, $S_{xx}$ and the EEG PSD written per
trial by `scripts/trfs/get_transfer_functions.py` -- so the regularisation, the normalisation and
the reference are all chosen here, and so is the envelope's unit: with `ENVELOPE_IN_PASCALS`
each stimulus's envelope is converted to Pa$^{0.6}$ from its calibrated presentation level. `exploration/transfer_functions.ipynb` is where those choices
were made and checked; this notebook only draws the result.

**Figure 6** -- the five bands of figure 2, one row each, the transfer function on the left and
its analytic envelope on the right.

## Setup

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from mne import pick_info
from mne.io import read_raw_fif
from mne.viz import plot_topomap
from mne.viz.evoked import _get_pos_outlines, _rgb
from mne.viz.topomap import _draw_outlines
from scipy.signal import firwin, freqz, hilbert, iirnotch, kaiserord

ROOT = Path.cwd().parent if Path.cwd().name == "figures" else Path.cwd()
CONFIG = json.loads((ROOT / "src" / "package_data" / "config.json").read_text())
sys.path.insert(0, str(ROOT / "src"))

from trfs.transfer_function import (
    align_columns,
    load_trial,
    rereference_transfer_function,
    resolve_trf_dir,
    transfer_function_from_parts,
)

# --- what is fitted ----------------------------------------------------------
REFERENCE = "avg"                       # "avg" | "linked_mastoids" | "FCz"
WINDOW_SECONDS, OVERLAP = 5.0, 0.8      # 60.0 is the other run on disk
FEATURE = "gammatone_envelope"
# In units of the stimulus spectrum's own mean. 0 is the plain quotient; the exploration
# notebook's section 1 has the damping factors per band for the alternative.
RIDGE = 0.0

TRF_DIR = resolve_trf_dir(CONFIG, REFERENCE, WINDOW_SECONDS, OVERLAP, ROOT)
EEG_DIR = Path(CONFIG["eeg_dir"])
OUT_DIR = ROOT / "figures"
CACHE_ROOT = OUT_DIR / "figure_data"

TEMPLATE_EEG = EEG_DIR / "sub-001_-_ses-001_-_trial-001_-_cond-natural_-_audio01-audiobook_5_1_-_eeg.fif"
RAW_INFO = read_raw_fif(TEMPLATE_EEG, preload=False, verbose="ERROR").pick("eeg").info

paths = sorted(TRF_DIR.glob("*_-_eeg.npz"))
if not paths:
    raise FileNotFoundError(f"no trial files in {TRF_DIR}")

# The transform length, sampling rate and frequency grid are properties of the run, not of this
# notebook. `TrialSpectra` does not carry the first two, so they come straight off the file --
# derived from WINDOW_SECONDS they would silently disagree with the data if the run differed.
frequencies = load_trial(paths[0], FEATURE).frequencies
with np.load(paths[0]) as saved:
    fs = float(saved["fs"])
    nperseg = int(saved["nperseg"])
    assert float(saved["window_seconds"]) == WINDOW_SECONDS, (
        f"{TRF_DIR.name} holds a {float(saved['window_seconds']):g} s window, "
        f"not the {WINDOW_SECONDS:g} s this notebook asks for")
assert len(frequencies) == nperseg // 2 + 1, "frequency grid does not match the transform"
# Zero lag in the middle of the transform, which is circular with the window's period.
times = (np.arange(nperseg) - nperseg // 2) / fs * 1000

# --- house style, shared with figures 1-5 ------------------------------------
DEFAULT_FONTSIZE = 10
TICK_FONTSIZE = 8
SMALL_FONTSIZE = 6.5
PANEL_LABEL_FONTSIZE = 14

INK, MUTED, HAIRLINE = "#1b1b1a", "#6b6b68", "#d5d5d1"

plt.rcParams.update({
    "font.size": TICK_FONTSIZE,
    "axes.labelsize": DEFAULT_FONTSIZE,
    "axes.titlesize": DEFAULT_FONTSIZE,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": TICK_FONTSIZE,
    "axes.linewidth": 0.8,
    "figure.dpi": 130,
    # TrueType, not matplotlib's default Type 3, which publishers flag.
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def bare(ax):
    """Recessive chrome: drop the top/right rules."""
    ax.spines[["top", "right"]].set_visible(False)
    return ax


print(f"{len(paths)} trials in {TRF_DIR.name} under {REFERENCE}, "
      f"{sum(p.stat().st_size for p in paths) / 1e9:.1f} GB")
print(f"{len(frequencies)} bins at {frequencies[1] - frequencies[0]:g} Hz; "
      f"window {nperseg} samples ({nperseg / fs:g} s) at {fs:g} Hz")

## The envelope in physical units

The envelope is stored in the audio files' digital units, and those do not reflect what
was heard. The files sit at a handful of different RMS levels (-28 to -20 dBFS), but every
stimulus was presented at **62 dB(A)**. The dataset calibrated each one with speech-shaped
noise of the same spectrum and RMS, in a 2 cm$^3$ coupler (Accou et al.). So each
stimulus's digital-to-pressure factor is known from its own level:

$$k_i = \frac{p_{62}}{r_{A,i}}, \qquad p_{62} = 20\,\mu\mathrm{Pa} \cdot 10^{62/20} \approx 0.025\ \mathrm{Pa}$$

where $r_{A,i}$ is the stimulus's A-weighted RMS in digital full-scale units, over the
whole file. The noise had the stimulus's spectrum and RMS, so its A-weighted level, which
was what the meter read, is the stimulus's own long-term A-weighted level.

The envelope is not linear in the waveform. Each gammatone subband is half-wave rectified
and raised to the power 0.6 before they are summed. So the envelope scales as
$k_i^{0.6}$, and a trial's transfer function converts as $H_i / k_i^{0.6}$. That puts it in
V per Pa$^{0.6}$, coupler-referenced, up to the filterbank's fixed gain, which is common
to every stimulus. `ENVELOPE_IN_PASCALS` applies this per trial in the pooling pass. Being
per stimulus, it is not a global rescaling: it also changes how trials are weighted
against each other. Podcast files sit ~6 dB hotter than audiobook files, so in digital
units their trials counted for less.

`CALIBRATION_TUBE_FILTER` A-weights the audio after the same insert-tube low-pass the
feature pipeline applies, which approximates what the coupler measured through the ER-3A.
Both variants are computed and cached; the cell prints how far they differ. What stays
uncertain: the level is coupler- rather than ear-referenced, and 62 dB(A) is the
long-term level, silences included.

The time-domain TRF is the inverse transform of $H$ on the fit's grid, so strictly it is
per sample at the envelope rate; the figures label it per Pa$^{0.6}$, as is conventional.

In [ ]:
import re

from scipy.signal import bilinear_zpk, sosfilt, zpk2sos

from audio_features.factory import (DEFAULT_LOWPASS, Brian2HearsGammatoneCochleogramExtractor,
                                    read_wav_mono)

# --- the envelope in physical units -------------------------------------------------------
# True rescales every trial's transfer function from digital envelope units to Pa^0.6 (see above).
ENVELOPE_IN_PASCALS = True
PRESENTATION_LEVEL_DBA = 62.0
REFERENCE_PRESSURE_PA = 20e-6
# A-weight the tube-filtered audio (as the feature pipeline sees it) rather than the raw file.
CALIBRATION_TUBE_FILTER = True
# The extractor the stored envelopes were made with, and so its compression exponent.
ENVELOPE_COMPRESSION = Brian2HearsGammatoneCochleogramExtractor.compression
RAW_WAVES_DIR = Path(CONFIG["raw_waves_dir"])
PRESENTATION_PRESSURE_PA = REFERENCE_PRESSURE_PA * 10 ** (PRESENTATION_LEVEL_DBA / 20)


def a_weighting_sos(fs):
    """IEC 61672 A-weighting as second-order sections, by the bilinear transform; 0 dB at 1 kHz."""
    f1, f2, f3, f4 = 20.598997, 107.65265, 737.86223, 12194.217
    poles = [-2 * np.pi * f4] * 2 + [-2 * np.pi * f3, -2 * np.pi * f2] + [-2 * np.pi * f1] * 2
    gain = (2 * np.pi * f4) ** 2 * 10 ** (1.9997 / 20)
    return zpk2sos(*bilinear_zpk([0, 0, 0, 0], poles, gain, fs))


def a_weighted_rms(audio, fs):
    return float(np.sqrt(np.mean(sosfilt(a_weighting_sos(fs), audio) ** 2)))


def stimulus_calibration(stimuli):
    """Each stimulus's A-weighted digital RMS, raw and tube-filtered, cached once for all
    references: it depends only on the audio files."""
    cache = CACHE_ROOT / "stimulus_pressure_calibration.npz"
    if cache.exists():
        with np.load(cache) as saved:
            known = {str(name): (float(raw), float(tube)) for name, raw, tube
                     in zip(saved["stimuli"], saved["rms_a_raw"], saved["rms_a_tube"])}
        if set(stimuli) <= set(known):
            return known
        print("calibration cache lacks some stimuli, recomputing")
    known = {}
    for count, name in enumerate(stimuli, 1):
        audio, fs = read_wav_mono(RAW_WAVES_DIR / f"{name}.wav")
        tube, tube_fs = DEFAULT_LOWPASS(audio, fs)
        known[name] = (a_weighted_rms(audio, fs), a_weighted_rms(tube, tube_fs))
        if count % 10 == 0 or count == len(stimuli):
            print(f"  {count}/{len(stimuli)} stimuli", flush=True)
    names = sorted(known)
    np.savez(cache, stimuli=np.array(names),
             rms_a_raw=np.array([known[n][0] for n in names]),
             rms_a_tube=np.array([known[n][1] for n in names]))
    return known


# The stimulus of every trial, from its file name.
TRIAL_STIMULUS = {path: re.search(r"_-_audio\d+-(.+)_-_eeg$", path.stem).group(1)
                  for path in paths}
calibration = stimulus_calibration(sorted(set(TRIAL_STIMULUS.values())))
rms_a = {name: values[1] if CALIBRATION_TUBE_FILTER else values[0]
         for name, values in calibration.items()}
# Pa per digital full-scale unit, and what a trial's H is divided by.
PRESSURE_FACTOR = {name: PRESENTATION_PRESSURE_PA / value for name, value in rms_a.items()}
ENVELOPE_FACTOR = {name: k ** ENVELOPE_COMPRESSION for name, k in PRESSURE_FACTOR.items()}

levels = np.array([20 * np.log10(rms_a[name]) for name in sorted(rms_a)])
difference = np.array([20 * np.log10(calibration[n][1] / calibration[n][0]) for n in sorted(rms_a)])
print(f"{len(rms_a)} stimuli; A-weighted level {levels.min():.1f} to {levels.max():.1f} dBFS "
      f"({'tube-filtered' if CALIBRATION_TUBE_FILTER else 'raw'}); tube filter changes it by "
      f"{difference.min():+.2f} to {difference.max():+.2f} dB")
print(f"62 dB(A) = {PRESENTATION_PRESSURE_PA * 1e3:.1f} mPa; envelope factor k^{ENVELOPE_COMPRESSION:g} "
      f"spans {min(ENVELOPE_FACTOR.values()):.3f}-{max(ENVELOPE_FACTOR.values()):.3f} "
      f"({20 * np.log10(max(ENVELOPE_FACTOR.values()) / min(ENVELOPE_FACTOR.values())):.1f} dB)")
print(f"envelope in Pa^{ENVELOPE_COMPRESSION:g}: {'ON' if ENVELOPE_IN_PASCALS else 'off (digital units)'}")

## The bands, and the filters that cut them

Figure 2's five bands. Each is imposed by a Kaiser-window FIR designed with `kaiserord`,
evaluated on the fit's own frequency grid and multiplied onto $H$ -- not by zeroing bins, which
is an ideal rectangular response whose impulse response rings for the whole window. The
**magnitude** of the response is applied rather than the complex response, which makes it zero
phase: latency is what this figure measures, so a group delay would bias the one quantity being
read.

Every mains harmonic up to Nyquist is notched, each at fixed bandwidth rather than fixed Q.

The transition width scales to the band, and that is where the window binds. The tap count goes
as 1/width, and a kernel longer than the period the transform is circular over rings round into
itself, so the design is capped at the window length and the transition widens instead. What
that costs is **stopband attenuation**, not edge position -- `firwin` puts the $-6$ dB points at
the cutoffs whatever the tap count -- so the table below reports the worst out-of-band response
and the rejection at DC, and flags any band that is leaking.

In [ ]:
# --- the bands ---------------------------------------------------------------
BANDS = [(0.4, 2.0), (4.0, 15.0), (30.0, 60.0), (80.0, 120.0), (140.0, 250.0)]
BAND_NAMES = ["Delta", "Theta-alpha", "Low gamma", "Mid gamma", "High gamma"]
# The whole range the fit covers, for figure 7. Pooled in the same pass and cached in the same
# file as the five, so the two figures cannot end up built from different trials.
#
# 0.2-300 Hz with a flat 1 Hz transition, exactly as exploration/transfer_functions.ipynb
# section 6 fits it. Note what the table below then reports: in a 5 s window this rejects DC by
# only 9 dB, so the broadband row carries an offset at every lag. Raising the lower edge to
# 0.5 Hz would take that to -55 dB, and the 60 s run would fix it outright -- both are changes
# to the settled pipeline, so neither is made here.
BROADBAND = (0.2, 300.0)
# The spectrum cut in two, for figure 8: the cortical range and everything above it.
SPLIT_BANDS = [(0.2, 30.0), (30.0, 250.0)]
# The per-frequency standardised figure: the cortical range, the high range, and both. Pooled
# with each band's own EEG SD, as figures 6 and 8 are. The lower edge is 1 Hz rather than
# 0.2 Hz because standardising tilts the spectrum towards its lowest bins, where a 0.2 Hz edge
# passes DC and drift in a 5 s window. 0.5 Hz was tried: it rejects DC by ~55 dB, but the
# standardised traces then carry per-channel offsets and slow drift across the whole window.
STANDARDISED_BANDS = [(1.0, 30.0), (30.0, 250.0), (1.0, 250.0)]
# Only those not already pooled for another figure: 30-250 Hz is figure 8's high half.
EXTRA_BANDS = [band for band in STANDARDISED_BANDS if band not in BANDS + [BROADBAND] + SPLIT_BANDS]
ALL_BANDS = BANDS + [BROADBAND] + SPLIT_BANDS + EXTRA_BANDS
ALL_NAMES = (BAND_NAMES + ["Broadband", "Low split", "High split"]
             + [f"Standardised {low:g}-{high:g} Hz" for low, high in EXTRA_BANDS])

# --- the filters -------------------------------------------------------------
FILTER_RIPPLE_DB = 60.0
MAX_TRANSITION_HZ = 1.0         # the ceiling; each band may want narrower
# Bands that do not want the `low / 2` rule below. The broadband is set to the flat 1 Hz the
# exploration notebook used; without this entry the rule would give it 0.1 Hz, which is a
# different filter from the one that notebook settled on.
# 0.2-30 Hz gets the same treatment as the broadband for the same reason -- and it is a
# choice with a cost either way, so both sides are measured. At the rule's 0.1 Hz the kernel is
# capped and the stopband is -15 dB everywhere; at a flat 1 Hz it is -67 dB everywhere except
# DC, which goes from -15 to -9 dB. Better rejection of real signal, slightly worse rejection
# of an offset that no lower edge at 0.2 Hz can remove in a 5 s window anyway.
# The 1 Hz-edged bands likewise: the rule's 0.5 Hz would want ~7400 taps, past the 5 s cap.
TRANSITION_OVERRIDE = {BROADBAND: 1.0, (0.2, 30.0): 1.0, (1.0, 30.0): 1.0, (1.0, 250.0): 1.0}
MAINS_HZ, NOTCH_BANDWIDTH_HZ = 50.0, 2.0
# A stopband shallower than this means the row is showing content from outside its own band.
STOPBAND_WARN_DB = -40.0


def notch_response():
    """Every mains harmonic up to Nyquist, as a product of `iirnotch` magnitudes.

    Fixed bandwidth rather than fixed Q: `iirnotch` takes Q, and a notch's width is f0/Q, so a
    constant Q would widen every harmonic in proportion to its frequency.
    """
    response = np.ones_like(frequencies)
    harmonics = np.arange(MAINS_HZ, fs / 2, MAINS_HZ)
    for harmonic in harmonics:
        b, a = iirnotch(harmonic, harmonic / NOTCH_BANDWIDTH_HZ, fs=fs)
        response *= np.abs(freqz(b, a, worN=frequencies, fs=fs)[1])
    return response, harmonics


def band_filter(low, high, transition=None):
    """`(magnitude on the fit's grid, taps used, taps wanted, transition)` for one band.

    `transition` (Hz) overrides the rule below for this call only.
    """
    # Narrow enough not to eat the band, and not so narrow at the low edge that the passband
    # would start below DC -- unless the band says otherwise.
    if transition is None:
        transition = TRANSITION_OVERRIDE.get(
            (low, high), min(MAX_TRANSITION_HZ, low / 2, (high - low) / 4))
    wanted, beta = kaiserord(FILTER_RIPPLE_DB, transition / (fs / 2))
    used = min(wanted, nperseg - 1) | 1          # odd -> type I, exactly linear phase
    taps = firwin(used, [low, high], window=("kaiser", beta), pass_zero=False, fs=fs)
    return np.abs(freqz(taps, worN=frequencies, fs=fs)[1]), used, wanted, transition


def filter_quality(response, low, high, transition):
    """`(worst stopband dB, where, DC dB)` -- what actually degrades when taps run out."""
    decibels = 20 * np.log10(response + 1e-15)
    stop = (frequencies < low - transition) | (frequencies > high + transition)
    worst = int(np.argmax(decibels[stop]))
    return decibels[stop][worst], frequencies[stop][worst], decibels[0]


NOTCHES, HARMONICS = notch_response()
BAND_RESPONSES, BAND_QUALITY = {}, {}
for band in ALL_BANDS:
    response, used, wanted, transition = band_filter(*band)
    BAND_RESPONSES[band] = response * NOTCHES
    BAND_QUALITY[band] = (used, wanted, transition) + filter_quality(response, *band, transition)

LEAKY = {band: BAND_QUALITY[band][3] > STOPBAND_WARN_DB for band in ALL_BANDS}

print(f"{len(HARMONICS)} mains harmonics notched, {NOTCH_BANDWIDTH_HZ:g} Hz wide")
print(f"kernels capped at {nperseg - 1} taps by the {nperseg / fs:g} s window\n")
print(f"{'band':>16s} {'taps used/wanted':>18s} {'transition':>11s} "
      f"{'worst stopband':>18s} {'at DC':>10s}")
for band, name in zip(ALL_BANDS, ALL_NAMES):
    used, wanted, transition, worst, where, dc = BAND_QUALITY[band]
    flag = ("  <- LEAKY" if LEAKY[band] else "  capped" if used < wanted else "")
    print(f"{f'{band[0]:g}-{band[1]:g} Hz':>16s} {f'{used}/{wanted}':>18s} "
          f"{transition:8.3f} Hz {worst:+10.1f} @ {where:5.1f} {dc:+9.1f} dB{flag}")
if any(LEAKY.values()):
    names = [f"{b[0]:g}-{b[1]:g} Hz" for b in ALL_BANDS if LEAKY[b]]
    print(f"\n{', '.join(names)} cannot be realised in a {nperseg / fs:g} s window. "
          f"Set WINDOW_SECONDS = 60.0 in the setup cell to use the longer run.")

### The filter, drawn

The magnitude response of one band's filter, as it is multiplied onto $H$: the Kaiser FIR
times the mains notches, evaluated on the fit's own frequency grid. The top panel shows the
whole grid in dB, with the 60 dB design target dashed. The lower panel shows the band and
its edges on a linear scale, one point per frequency bin. `FILTER_PLOT_BAND` picks the band.

In [ ]:
# ===================== filter response plot ==================================
# Any (low, high): one of ALL_BANDS reads the response the figures use; anything else is designed
# here by the same rules (band_filter, with the same notches).
FILTER_PLOT_BAND = (0.4, 2)
FILTER_PLOT_FIGSIZE = (6.27, 4.2)
FILTER_PLOT_ZOOM = 6            # the lower panel spans the band +/- this many transition widths
FILTER_PLOT_FLOOR_DB = -120.0   # bottom of the dB axis
# =============================================================================

if FILTER_PLOT_BAND in BAND_RESPONSES:
    plot_response = BAND_RESPONSES[FILTER_PLOT_BAND]
    plot_used, plot_wanted, plot_transition = BAND_QUALITY[FILTER_PLOT_BAND][:3]
else:
    unnotched, plot_used, plot_wanted, plot_transition = band_filter(*FILTER_PLOT_BAND)
    plot_response = unnotched * NOTCHES
low, high = FILTER_PLOT_BAND
decibels = 20 * np.log10(plot_response + 1e-15)

fig, (ax_full, ax_zoom) = plt.subplots(2, 1, figsize=FILTER_PLOT_FIGSIZE,
                                       gridspec_kw=dict(hspace=0.55))
# The whole grid, in dB: passband, both transitions, stopband floor, and every mains notch.
ax_full.plot(frequencies, decibels, color=INK, lw=0.8)
ax_full.axhline(-FILTER_RIPPLE_DB, color=MUTED, lw=0.8, ls="--", zorder=0)
ax_full.axvspan(low, high, color=HAIRLINE, alpha=0.6, lw=0, zorder=0)
ax_full.set_xlim(0, fs / 2)
ax_full.set_ylim(FILTER_PLOT_FLOOR_DB, 10)
ax_full.set_ylabel("Gain (dB)")
ax_full.set_title(f"{low:g}-{high:g} Hz: Kaiser FIR, {plot_used} of {plot_wanted} taps, "
                  f"{plot_transition:g} Hz transition, mains notched", fontsize=TICK_FONTSIZE)
bare(ax_full)

# The band itself, linear: what the passband and its edges do to H, bin by bin on the fit's grid.
span = FILTER_PLOT_ZOOM * plot_transition
shown = (frequencies >= max(low - span, 0)) & (frequencies <= high + span)
ax_zoom.plot(frequencies[shown], plot_response[shown], color=INK, lw=0.8, marker=".", ms=2)
for edge in (low, high):
    ax_zoom.axvline(edge, color=MUTED, lw=0.8, ls="--", zorder=0)
ax_zoom.set_ylim(0, 1.08)
ax_zoom.set_xlabel("Frequency (Hz)")
ax_zoom.set_ylabel("Gain")
bare(ax_zoom)

stem = f"filter_response_{low:g}-{high:g}Hz_{WINDOW_SECONDS:g}s"
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight")
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", dpi=200)
plt.show()

passband = (frequencies >= low + plot_transition) & (frequencies <= high - plot_transition)
clean = passband & (NOTCHES > 0.99)
print(f"passband ripple {decibels[clean].min():+.3f} to {decibels[clean].max():+.3f} dB away "
      f"from the notches; gain at the edges {plot_response[np.argmin(np.abs(frequencies - low))]:.2f} "
      f"({low:g} Hz) and {plot_response[np.argmin(np.abs(frequencies - high))]:.2f} ({high:g} Hz); "
      f"applied as magnitude only, so zero phase")

## Pooling

Each trial fitted, then a duration-weighted average per participant, then the mean over
participants. Fitting per trial and averaging after -- rather than summing cross-spectra and
dividing once -- keeps every stimulus's own $S_{xx}$ in its own denominator.

Each channel is divided by the standard deviation of the EEG **that band's filter passes**, taken
from the PSD written for this reference. Without it $|H|$ is proportional to a channel's EEG
amplitude, so a topography maps where the EEG is large rather than where the response is, and the
grand average is led by whoever has the biggest signal. Per band rather than broadband because
EEG amplitude is strongly $1/f$: a broadband divisor is set almost entirely by the low end and
would weight the gamma rows' channels by their delta amplitude.

The pass is cached per reference, window, ridge and band set, so re-running the notebook is
free and changing any of them invalidates it.

In [ ]:
def cache_path():
    """One file per (reference, window, ridge, band set) -- a change to any of them is a miss."""
    key = json.dumps([REFERENCE, WINDOW_SECONDS, OVERLAP, FEATURE, RIDGE, ALL_BANDS,
                      FILTER_RIPPLE_DB, MAX_TRANSITION_HZ, MAINS_HZ, NOTCH_BANDWIDTH_HZ,
                      TRANSITION_OVERRIDE.get(BROADBAND), 2]
                     # The envelope's units, only when converted, so the digital-unit cache keeps
                     # its old key.
                     + ([["pascals", PRESENTATION_LEVEL_DBA, CALIBRATION_TUBE_FILTER,
                          ENVELOPE_COMPRESSION]] if ENVELOPE_IN_PASCALS else []),
                     sort_keys=True)
    directory = CACHE_ROOT / REFERENCE
    directory.mkdir(parents=True, exist_ok=True)
    return directory / f"trf_bands_{hashlib.sha1(key.encode()).hexdigest()[:12]}.npz"


def band_standard_deviation(psd, response):
    """Each channel's SD of the EEG `response` passes, from its one-sided PSD.

    By Parseval the variance is the integral of the PSD, so weighting the integrand by the
    filter's POWER response gives the SD of the filtered signal exactly -- band edges, roll-off
    and mains notches included.
    """
    return np.sqrt((psd * (response ** 2)[:, None]).sum(0) * (frequencies[1] - frequencies[0]))


def pool_bands():
    """Grand mean of H over participants, one array per band, in a single pass.

    Two stages collapsed into one accumulator per band. With every participant's total weight
    known in advance the two are the same sum::

        (1/S) * sum_s [ sum_t w_t X_t / W_s ]  ==  sum_{s,t} (1/S)(w_t / W_s) X_t

    which is what makes one pass enough for five bands at once rather than one accumulator per
    participant per band.
    """
    weights, axis = {}, []
    for path in paths:
        trial = load_trial(path, FEATURE)
        weights[trial.subject] = weights.get(trial.subject, 0.0) + trial.analysed_duration

    channels = None
    totals = {band: None for band in ALL_BANDS}
    # The broadband again, undivided, for the supplementary figure. Built in the SAME pass, so
    # the two versions cannot differ in anything except the divisor.
    raw_total = None
    scale = 1.0 / len(weights)
    hours = sum(weights.values()) / 3600

    for count, path in enumerate(paths, 1):
        trial = load_trial(path, FEATURE)
        try:
            cross, trial_channels = rereference_transfer_function(
                trial.cross_spectrum, trial.channel_names, REFERENCE
            )
        except ValueError:                  # no mastoids on this recording
            continue
        if channels is None:
            channels = list(trial_channels)
            for band in ALL_BANDS:
                totals[band] = np.zeros((len(frequencies), len(channels)), complex)
            raw_total = np.zeros((len(frequencies), len(channels)), complex)
        # The identity above assumes one channel axis throughout; with a ragged one the
        # per-column weights differ and the two sums are not the same. Checked, not assumed.
        assert trial_channels == channels, f"{path.stem}: unexpected channel axis"

        psd, psd_channels = trial.psd_for(REFERENCE)
        keep = [psd_channels.index(name) for name in trial_channels]
        H = transfer_function_from_parts(cross, trial.stimulus_psd, RIDGE)
        if ENVELOPE_IN_PASCALS:
            # Digital envelope units -> Pa^0.6: the envelope scales as k^0.6, so H as k^-0.6.
            H = H / ENVELOPE_FACTOR[TRIAL_STIMULUS[path]]
        share = scale * trial.analysed_duration / weights[trial.subject]
        for band in ALL_BANDS:
            sd = band_standard_deviation(psd, BAND_RESPONSES[band])[keep]
            totals[band] += share * (H / sd)
        raw_total += share * H
        if count % 100 == 0 or count == len(paths):
            print(f"  {count}/{len(paths)} trials", flush=True)

    return totals, raw_total, channels, len(weights), hours


cache = cache_path()
if cache.exists():
    with np.load(cache) as saved:
        BAND_FITS = {band: saved[f"band_{index}"] for index, band in enumerate(ALL_BANDS)}
        BROADBAND_RAW_FIT = saved["broadband_raw"]
        channels = [str(name) for name in saved["channel_names"]]
        n_subjects = int(saved["n_subjects"])
        listening_hours = float(saved["hours"])
    print(f"cached: {cache.name}")
else:
    BAND_FITS, BROADBAND_RAW_FIT, channels, n_subjects, listening_hours = pool_bands()
    np.savez_compressed(
        cache, channel_names=np.array(channels), n_subjects=n_subjects,
        hours=listening_hours, broadband_raw=BROADBAND_RAW_FIT,
        **{f"band_{index}": BAND_FITS[band] for index, band in enumerate(ALL_BANDS)},
    )
    print(f"written: {cache.name} ({cache.stat().st_size / 1e6:.0f} MB)")

INFO = pick_info(RAW_INFO, [RAW_INFO.ch_names.index(name) for name in channels])
# Position-to-colour, from the sensor layout: nearby channels get nearby colours, so a butterfly
# shows *where* a deflection is without needing 64 legend entries.
CHANNEL_COLOURS = _rgb(*np.array([channel["loc"][:3] for channel in INFO["chs"]]).T)

print(f"{REFERENCE}: {n_subjects} participants, {len(channels)} channels, "
      f"{listening_hours:.1f} listening hours; ridge {RIDGE:g}, per-band SD; envelope in "
      f"{f'Pa^{ENVELOPE_COMPRESSION:g}' if ENVELOPE_IN_PASCALS else 'digital units'}")

## Figure 6 -- the transfer function, band by band

One row per band. The left panel is every channel, coloured by where it sits on the head; the
right is the analytic envelope of those same traces.

The envelope is what carries latency. A band-limited transfer function is an oscillation whose
carrier phase is arbitrary, so its own peaks move with the band while the envelope's do not --
and because the filter is zero-phase, the envelope's peak is an unbiased estimate of the lag.
Its *precision* is set by the bandwidth: a band of width $w$ spreads a feature over roughly
$1/w$, so the narrow low rows localise less sharply than the wide high ones.

The lag window is per row, since a delta response is slow and a high-gamma one is over within a
few milliseconds of zero. The topography in each envelope panel is taken at that row's peak.

In [ ]:
# --- what the figure draws ---------------------------------------------------
# One lag window per band, and the ticks for it -- fixed rather than left to a locator, so the
# numbers are round and the rows do not each pick their own convention.
BAND_LAGS = [(-500.0, 1000.0), (-100.0, 500.0), (-50.0, 150.0), (-50.0, 150.0), (-50.0, 150.0)]
BAND_XTICKS = [[-500, 0, 500, 1000], [-100, 0, 200, 400], [-50, 0, 50, 100, 150],
               [-50, 0, 50, 100, 150], [-50, 0, 50, 100, 150]]

# Upper limit of each envelope panel. The transfer function panel beside it is then drawn
# symmetric on +/- the same number, so a row's two panels share one scale and the envelope can
# be read directly against the traces it is the envelope OF.
#
# `None` autoscales that row to its own envelope maximum, with `YLIM_HEADROOM` of air. The
# printed table gives the value each row settled on, ready to be pinned here.
BAND_YLIM = [7e-4, 5e-3, 5e-2, 8e-2, 6e-2]
YLIM_HEADROOM = 1.05

# Channels drawn on top of the rest, one entry per row -- `None` leaves that row alone. A row
# with a set also gets that set's OWN latency: its mean envelope peaks where the subset
# responds, which is not where the whole-scalp mean peaks once the subset is chosen because the
# response lives there. That latency gets a dotted rule of its own, against the dashed one.
CENTRAL_CLUSTER = [
    "CP1", "CPz", "CP2",
    "P1", "Pz", "P2",
]
HIGHLIGHT_TRF = [None, None, None, None, None]
HIGHLIGHT_ENVELOPE = [None, None, None, CENTRAL_CLUSTER, None]
# Raising the z-order alone moves a 0.5 pt line at alpha 0.55 in front of 63 others that look
# exactly like it, which is not visible. These are what make the subset readable; set them to
# the base values (0.5, 0.55) for pure z-order and nothing else.
HIGHLIGHT_WIDTH, HIGHLIGHT_ALPHA = 1.0, 1.0

FIGSIZE = (6.27, 7.1)
TOPO_RECT = [0.62, 0.48, 0.52, 0.52]    # topography inset, in the envelope panel
TOPO_WIDTH = 3                           # adjacent samples averaged at the peak
SENSOR_KEY_RECT = TOPO_RECT

assert (len(BAND_LAGS) == len(BAND_XTICKS) == len(BAND_YLIM) == len(HIGHLIGHT_TRF)
        == len(HIGHLIGHT_ENVELOPE) == len(BANDS)), \
    "one lag window, tick set, y limit and highlight set per band"


def band_response_in_lag(band):
    """H for one band, filtered and inverse-transformed: `(trf, envelope)`, both (n_lags, C)."""
    spectrum = BAND_FITS[band] * BAND_RESPONSES[band][:, None]
    trf = np.roll(np.fft.irfft(spectrum, n=nperseg, axis=0), nperseg // 2, axis=0)
    return trf, np.abs(hilbert(trf, axis=0))


def plot_sensor_key(ax):
    """The head that says which colour is where."""
    positions, outlines = _get_pos_outlines(INFO, np.arange(len(INFO.ch_names)),
                                            sphere=(0, 0, 0, 0.095))
    _draw_outlines(ax, outlines)
    ax.scatter(*positions.T, c=CHANNEL_COLOURS, s=7, linewidths=0)
    ax.set_aspect("equal")
    ax.set_axis_off()


def draw_highlight(ax, data, names, shown):
    """Redraw the named channels on top of the rest, keeping their position colour.

    Above the other traces but below the channel mean, which stays the panel's headline.
    """
    if not names:
        return
    missing = [name for name in names if name not in INFO.ch_names]
    assert not missing, f"highlight names not in the {REFERENCE} montage: {missing}"
    for name in names:
        index = INFO.ch_names.index(name)
        ax.plot(times[shown], data[shown, index], color=CHANNEL_COLOURS[index],
                lw=HIGHLIGHT_WIDTH, alpha=HIGHLIGHT_ALPHA, zorder=3)


fig = plt.figure(figsize=FIGSIZE)
# Most of the height saved is gutter, not panel: at 0.62 the gaps took more room than the
# rows did. The gutter still has to hold two things per boundary -- the upper row's tick
# numbers and the lower row's `1e-N` exponent, which matplotlib draws ABOVE its axes --
# so it cannot go much below this even with the x-labels gone from all but the last row.
grid = fig.add_gridspec(len(BANDS), 2, hspace=0.48, wspace=0.28)
peaks, limits, subset_latencies = [], [], []

for row, (band, name) in enumerate(zip(BANDS, BAND_NAMES)):
    trf, envelope = band_response_in_lag(band)
    mean_envelope = envelope.mean(-1)
    lags = BAND_LAGS[row]
    shown = (times >= lags[0]) & (times <= lags[1])
    peak = int(np.argmax(np.where(shown, mean_envelope, -np.inf)))
    peaks.append((band, name, times[peak], envelope[peak]))

    ax_trf = fig.add_subplot(grid[row, 0])
    ax_env = fig.add_subplot(grid[row, 1], sharex=ax_trf)
    for index in range(trf.shape[1]):
        ax_trf.plot(times[shown], trf[shown, index], color=CHANNEL_COLOURS[index],
                    lw=0.5, alpha=0.55, zorder=2)
        ax_env.plot(times[shown], envelope[shown, index], color=CHANNEL_COLOURS[index],
                    lw=0.5, alpha=0.55, zorder=2)
    draw_highlight(ax_trf, trf, HIGHLIGHT_TRF[row], shown)
    draw_highlight(ax_env, envelope, HIGHLIGHT_ENVELOPE[row], shown)
    ax_env.plot(times[shown], mean_envelope[shown], color=INK, lw=1.3, zorder=5)

    for ax in (ax_trf, ax_env):
        ax.axvline(0, color=HAIRLINE, lw=0.8, zorder=0)
        ax.axvline(times[peak], color="k", lw=0.8, ls="--", alpha=0.85, zorder=1)
        ax.set_xlim(*lags)
        ax.set_xticks(BAND_XTICKS[row])
        # Each row has its own x range, so every row keeps its tick numbers -- but the word
        # "Lag (ms)" ten times is ten times the height for one piece of information.
        if row == len(BANDS) - 1:
            ax.set_xlabel("Lag (ms)", fontsize=DEFAULT_FONTSIZE)
        # A common exponent rather than the decimals spelled out on every tick: at five rows
        # and two columns the long form runs one panel's numbers into the next.
        ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))
        ax.yaxis.get_offset_text().set_fontsize(SMALL_FONTSIZE)
        bare(ax)
    ax_trf.axhline(0, color=HAIRLINE, lw=0.8, zorder=0)
    # The envelope bounds the transfer function it came from, so one number sets both: the
    # left panel cannot clip on a limit its own envelope fits inside.
    limit = BAND_YLIM[row]
    if limit is None:
        limit = float(envelope[shown].max()) * YLIM_HEADROOM
    limits.append(limit)
    ax_env.set_ylim(0, limit)
    ax_trf.set_ylim(-limit, limit)
    # The band itself carries the row; the quantity is in the column titles and the names are
    # in the caption, where they can be argued with.
    ax_trf.set_ylabel(f"{band[0]:g}-{band[1]:g} Hz", fontsize=DEFAULT_FONTSIZE,
                      fontweight="bold")
    ax_env.annotate(f"{times[peak]:.0f} ms", (times[peak], 0.97),
                    xycoords=("data", "axes fraction"), textcoords="offset points",
                    xytext=(3, 0), fontsize=SMALL_FONTSIZE, color="k", va="top")

    def envelope_at(lag_index):
        """The envelope averaged over `TOPO_WIDTH` samples centred on a lag, clipped at the ends.

        Clipped rather than wrapped: the lag axis is circular with the window's period, so a
        window running off the edge would come back a whole period away from what it claims.
        """
        start = min(max(lag_index - TOPO_WIDTH // 2, 0), len(times) - TOPO_WIDTH)
        return envelope[start:start + TOPO_WIDTH].mean(0)

    highlight_names = HIGHLIGHT_ENVELOPE[row]
    subset_latency = None
    if highlight_names:
        index = [INFO.ch_names.index(name) for name in highlight_names]
        subset = envelope[:, index].mean(-1)
        subset_peak = int(np.argmax(np.where(shown, subset, -np.inf)))
        subset_latency = times[subset_peak]
        # Dotted and grey, against the whole-scalp peak's dashed black rule -- two differences
        # rather than one, so the pair is told apart at a glance rather than by position.
        ax_env.axvline(subset_latency, color=MUTED, lw=0.9, ls=":", alpha=0.95, zorder=4)
        # To the LEFT of its rule, where the whole-scalp label sits to the right of its own --
        # the two latencies are close enough that both on the same side would collide.
        ax_env.annotate(f"{subset_latency:.0f} ms", (subset_latency, 0.97),
                        xycoords=("data", "axes fraction"), textcoords="offset points",
                        xytext=(-3, 0), fontsize=SMALL_FONTSIZE, color=MUTED,
                        va="top", ha="right")
    subset_latencies.append(subset_latency)

    # One map per panel, at the whole-scalp peak. Sequential and from zero: the envelope is
    # non-negative.
    values = envelope_at(peak)
    topo_ax = ax_env.inset_axes(TOPO_RECT)
    plot_topomap(values, INFO, axes=topo_ax, show=False, cmap="viridis",
                 vlim=(0, values.max()), contours=0, sensors=False)

    if row == 0:
        ax_trf.set_title("Narrowband TRF", fontsize=DEFAULT_FONTSIZE)
        ax_env.set_title("Envelope of TRF", fontsize=DEFAULT_FONTSIZE)
        plot_sensor_key(ax_trf.inset_axes(SENSOR_KEY_RECT))

# One marker per row rather than per panel: the two panels are one result seen two ways.
for row, ax in enumerate(fig.axes[:2 * len(BANDS):2]):
    ax.text(-0.26, 1.02, f"{'abcde'[row]})", transform=ax.transAxes,
            fontsize=PANEL_LABEL_FONTSIZE, fontweight="bold", va="bottom", ha="left")

# No suptitle: the reference, participant count and listening time belong in the caption, and
# the cells above print them.
stem = f"trf_bands_{REFERENCE}"
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

print(f"{'band':>16s} {'name':>14s} {'lag window':>18s} {'peak':>9s} "
      f"{'y limit':>10s}  strongest")
for (band, name, latency, values), limit in zip(peaks, limits):
    row = BANDS.index(band)
    flag = "   (LEAKY -- see the filter table)" if LEAKY[band] else ""
    print(f"{f'{band[0]:g}-{band[1]:g} Hz':>16s} {name:>14s} "
          f"{f'{BAND_LAGS[row][0]:g} to {BAND_LAGS[row][1]:g} ms':>18s} "
          f"{latency:6.1f} ms {limit:10.2e}  {INFO.ch_names[int(np.argmax(values))]}{flag}")
print("\nBAND_YLIM = [" + ", ".join(f"{limit:.1e}" for limit in limits)
      + "]   # the values above, to pin them")
for row, names in enumerate(HIGHLIGHT_ENVELOPE):
    if names:
        print(f"{'abcde'[row]}) envelope: {len(names)} channels brought forward "
              f"({', '.join(names)}); their own peak at {subset_latencies[row]:.1f} ms "
              f"against {peaks[row][2]:.1f} ms for the whole scalp")
for row, names in enumerate(HIGHLIGHT_TRF):
    if names:
        print(f"{'abcde'[row]}) transfer function: {len(names)} channels brought forward")

## Figure 7 -- the broadband transfer function

The old repo's figure 7 arrangement, over `FIGURE7_BAND` (the gamma range, 30-250 Hz, by
default): the transfer function on top, its analytic envelope underneath, and topographies hung
at chosen lags in each. The EEG normalisation is that band's own SD, as for every band figure.

The two panels carry different information. The transfer function is signed and oscillates, so
its topographies show the scalp reversing polarity from one lag to the next -- which is what the
`RdBu_r` maps in (a) are of. The envelope is non-negative and has no carrier, so its maps show
where the response has power irrespective of which way any channel is pointing at that instant.

Lags, widths and inset positions are set per panel below; figure 7 placed its by hand for the
same reason, since the response leaves different parts of the panel free at different lags.

In [ ]:
# ===================== figure 7 options ======================================
# The band figure 7 and its unnormalised supplement are drawn over. The gamma range by default:
# the manuscript's broadband TRF resolves the brainstem and midlatency components there.
# BROADBAND (0.2-300 Hz) draws the whole range the fit covers. It must be in ALL_BANDS, which
# is what pools it with its own EEG SD.
FIGURE7_BAND = (30.0, 250.0)
BROADBAND_LAGS = (-50.0, 100.0)
BROADBAND_XTICKS = [-50, 0, 50, 100]
# Upper limit of the envelope panel; the transfer function panel is symmetric on +/- the same
# number. `None` autoscales to the envelope with the headroom below -- more than the five-band
# figure needs, because the topographies hang inside the panel here rather than under it.
BROADBAND_YLIM = None
# Measured: the most extreme trace reaches 0.98 of the envelope maximum, but only briefly and
# only in the middle of the panel, where no map is placed. So the headroom can be cut well below
# what that single excursion would need -- the maps sit to either side of it, not above it.
BROADBAND_HEADROOM = 1.25

# Where the maps are taken, and where they sit. One (x, y) in axes fraction per lag, placed by
# hand as figure 7's were: the rules cluster in the middle of the panel, so the maps are parked
# to either side of that cluster and the last one drops to a second shelf.
#
# `TOPO_W` and `TOPO_H` are separate because the panel is about three times wider than it is
# tall. `plot_topomap` keeps the head square, so it draws at whichever of the two is smaller in
# inches -- a square 0.16 x 0.16 box is 0.86 in wide but only 0.30 in tall, which is why equal
# fractions gave heads a third the size of the sensor key beside them.
#
# The HEIGHT is what sets the drawn size here; the width only has to stay above it, and is kept
# narrow so three maps fit side by side inside the panel rather than running off its right.
TOPO_W, TOPO_H = 0.33, 0.33

TRF_TOPO_LAGS = [5.9, 8.8, 17.6, 23.4, 30.3]
TRF_TOPO_WIDTH = 3                       # adjacent samples averaged at each lag
# NOTE on the x values: these position the inset BOX, but the head is drawn square and
# centred inside it, so with TOPO_W wider than TOPO_H the box is mostly empty margin. At
# (0.33, 0.33) the box is 1.60 in wide and the head only 0.52 in, which is +/-0.053 of the
# panel width either side of the box centre -- so a head sits at `x + TOPO_W / 2`, and what has
# to stay inside [0, 1] is that centre plus 0.053, not the box.
TRF_TOPO_LOCS = [(-0.055, 0.60), (0.075, 0.60),
                 (0.515, 0.60), (0.645, 0.60), (0.775, 0.60)]

ENVELOPE_TOPO_LAGS = [6.8, 23.4]
ENVELOPE_TOPO_WIDTH = 3
# The early map to the far left, where the envelope is still flat, and the later two together
# on the right past the last rule.
ENVELOPE_TOPO_LOCS = [(-0.055, 0.58), (0.775, 0.58)]
BROADBAND_FIGSIZE = (6.27, 4.7)
# The same size as the maps and on the same left edge as the first of them, derived rather than
# repeated so it stays aligned if TOPO_W, TOPO_H or that first position changes. Its y puts it
# on the lower shelf, level with the 42 ms map at the other end of the panel.
BROADBAND_KEY_RECT = [TRF_TOPO_LOCS[0][0], 0.04, TOPO_W, TOPO_H]
# =============================================================================

assert FIGURE7_BAND in ALL_BANDS, f"{FIGURE7_BAND} is not pooled; add it to the band list"
assert len(TRF_TOPO_LAGS) == len(TRF_TOPO_LOCS), "one position per transfer-function lag"
assert len(ENVELOPE_TOPO_LAGS) == len(ENVELOPE_TOPO_LOCS), "one position per envelope lag"

broadband_trf, broadband_envelope = band_response_in_lag(FIGURE7_BAND)
broadband_mean = broadband_envelope.mean(-1)
shown = (times >= BROADBAND_LAGS[0]) & (times <= BROADBAND_LAGS[1])

limit = BROADBAND_YLIM
if limit is None:
    limit = float(broadband_envelope[shown].max()) * BROADBAND_HEADROOM

fig = plt.figure(figsize=BROADBAND_FIGSIZE)
grid = fig.add_gridspec(2, 1, hspace=0.30)
ax_trf = fig.add_subplot(grid[0])
ax_env = fig.add_subplot(grid[1], sharex=ax_trf)

for index in range(broadband_trf.shape[1]):
    ax_trf.plot(times[shown], broadband_trf[shown, index], color=CHANNEL_COLOURS[index],
                lw=0.5, alpha=0.55, zorder=2)
    ax_env.plot(times[shown], broadband_envelope[shown, index], color=CHANNEL_COLOURS[index],
                lw=0.5, alpha=0.55, zorder=2)
ax_env.plot(times[shown], broadband_mean[shown], color=INK, lw=1.3, zorder=5)

for ax in (ax_trf, ax_env):
    ax.axvline(0, color=HAIRLINE, lw=0.8, zorder=0)
    ax.set_xlim(*BROADBAND_LAGS)
    ax.set_xticks(BROADBAND_XTICKS)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))
    ax.yaxis.get_offset_text().set_fontsize(SMALL_FONTSIZE)
    bare(ax)
ax_trf.axhline(0, color=HAIRLINE, lw=0.8, zorder=0)
ax_trf.set_ylim(-limit, limit)
ax_env.set_ylim(0, limit)
ax_trf.set_ylabel("Activity (n.u.)")
ax_env.set_ylabel("Activity (n.u.)")
ax_env.set_xlabel("Lag (ms)")
ax_trf.set_title(f"TRF ({FIGURE7_BAND[0]:g}-{FIGURE7_BAND[1]:g} Hz)",
                 fontsize=DEFAULT_FONTSIZE)
ax_env.set_title("Envelope of TRF", fontsize=DEFAULT_FONTSIZE)


def hang_topographies(ax, data, lags, locations, width, cmap, symmetric):
    """A rule at each lag and a map above it, positioned by hand.

    `symmetric` picks the limits: the transfer function reverses sign across the scalp and
    needs a diverging map about zero, the envelope is non-negative and does not.
    """
    for lag, (x, y) in zip(lags, locations):
        start = min(max(int(np.argmin(np.abs(times - lag))) - width // 2, 0),
                    len(times) - width)
        values = data[start:start + width].mean(0)
        ax.axvline(lag, color="k", lw=0.7, ls="--", alpha=0.75, zorder=1)
        topo_ax = ax.inset_axes([x, y, TOPO_W, TOPO_H])
        extreme = float(np.abs(values).max())
        plot_topomap(values, INFO, axes=topo_ax, show=False, cmap=cmap,
                     vlim=(-extreme, extreme) if symmetric else (0, extreme),
                     contours=0, sensors=False)
        topo_ax.set_title(f"{lag:g} ms", fontsize=SMALL_FONTSIZE, pad=2)


hang_topographies(ax_trf, broadband_trf, TRF_TOPO_LAGS, TRF_TOPO_LOCS, TRF_TOPO_WIDTH,
                  "RdBu_r", symmetric=True)
hang_topographies(ax_env, broadband_envelope, ENVELOPE_TOPO_LAGS, ENVELOPE_TOPO_LOCS,
                  ENVELOPE_TOPO_WIDTH, "viridis", symmetric=False)
plot_sensor_key(ax_trf.inset_axes(BROADBAND_KEY_RECT))

for ax, label in [(ax_trf, "a)"), (ax_env, "b)")]:
    ax.text(-0.09, 1.02, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

stem = f"trf_broadband_{FIGURE7_BAND[0]:g}-{FIGURE7_BAND[1]:g}Hz_{REFERENCE}"
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

used, wanted, transition, worst, where, dc = BAND_QUALITY[FIGURE7_BAND]
print(f"{FIGURE7_BAND[0]:g}-{FIGURE7_BAND[1]:g} Hz, ridge {RIDGE:g}, {n_subjects} participants; "
      f"y limit {limit:.2e}")
print(f"  filter: {used}/{wanted} taps, {transition:g} Hz transition, "
      f"stopband {worst:+.1f} dB, DC {dc:+.1f} dB"
      + ("   <- DC is in the passband; see the band table" if LEAKY[FIGURE7_BAND] else ""))
peak = int(np.argmax(np.where(shown, broadband_mean, -np.inf)))
print(f"  channel-mean envelope peaks at {times[peak]:.1f} ms "
      f"(strongest {INFO.ch_names[int(np.argmax(broadband_envelope[peak]))]})")


def envelope_peaks(mean_envelope, window=(0.0, 60.0), count=2):
    """The `count` highest local maxima of a channel-mean envelope inside `window`, by lag."""
    from scipy.signal import find_peaks
    inside = np.flatnonzero((times >= window[0]) & (times <= window[1]))
    found, _ = find_peaks(mean_envelope[inside])
    top = found[np.argsort(mean_envelope[inside][found])[::-1][:count]]
    return sorted(times[inside][top])


print("  channel-mean envelope local maxima, 0-60 ms: "
      + ", ".join(f"{lag:.1f} ms" for lag in envelope_peaks(broadband_mean)))
for band, name in zip(BANDS, BAND_NAMES):
    if band[0] >= 30:
        band_mean = band_response_in_lag(band)[1].mean(-1)
        window = (times >= 0) & (times <= 60)
        print(f"  {name} ({band[0]:g}-{band[1]:g} Hz) envelope peak: "
              f"{times[window][np.argmax(band_mean[window])]:.1f} ms")

### Supplementary -- figure 7 without the normalisation

The same figure, over the same band, drawn from the pooled fit before the $\div$ SD step: same trials, same weights,
same filter, same lags, same channel axis, built in the same pass. Only the divisor is gone.

What changes and what does not:

* **Almost nothing at any one channel.** Within a trial the divisor is one number per channel,
  which rescales that channel's trace without bending it. Across trials it varies, so the pooled
  average is a differently weighted combination of the same trials and a feature *can* move --
  the cell measures how much rather than asserting it does not.
* **Everything across channels.** $|H|$ is proportional to a channel's EEG amplitude, and
  peripheral electrodes are large, so the butterfly is led by the rim of the montage and the
  topographies map *where the EEG is big* rather than where the response is.
* **Everything across participants.** The divisor is per trial, so it equalises participants as
  well as channels. Without it the grand average leans towards whoever has the largest signal,
  on top of the duration weighting that is intended.

The centre-to-rim ratios printed below are the number to read: the same quantity, drawn the same
way, with and without the step that makes a topography mean something.

In [ ]:
# ================= supplementary figure options ==============================
# Default to figure 7's settings so the two are comparable; change them here to pull them apart.
SUPP_LAGS = BROADBAND_LAGS
SUPP_XTICKS = list(BROADBAND_XTICKS)
SUPP_YLIM = None
SUPP_HEADROOM = BROADBAND_HEADROOM

SUPP_TRF_TOPO_LAGS = list(TRF_TOPO_LAGS)
SUPP_TRF_TOPO_LOCS = list(TRF_TOPO_LOCS)
SUPP_ENVELOPE_TOPO_LAGS = list(ENVELOPE_TOPO_LAGS)
SUPP_ENVELOPE_TOPO_LOCS = list(ENVELOPE_TOPO_LOCS)

# Midline against the rim, for the printed comparison.
CENTRE_SENSORS = ["Fz", "FCz", "Cz", "CPz", "Pz"]
RIM_SENSORS = ["Fp1", "Fp2", "O1", "O2", "P9", "P10", "T7", "T8"]
# =============================================================================

assert len(SUPP_TRF_TOPO_LAGS) == len(SUPP_TRF_TOPO_LOCS), "one position per lag"
assert len(SUPP_ENVELOPE_TOPO_LAGS) == len(SUPP_ENVELOPE_TOPO_LOCS), "one position per lag"

# Same two lines `band_response_in_lag` runs, on the undivided fit. With the envelope in Pa^0.6
# the undivided fit is in V per Pa^0.6 (the EEG is stored in volts), drawn here in microvolts.
SUPP_UNIT_SCALE = 1e6 if ENVELOPE_IN_PASCALS else 1.0
SUPP_UNIT_LABEL = (rf"TRF ($\mu$V Pa$^{{-{ENVELOPE_COMPRESSION:g}}}$)" if ENVELOPE_IN_PASCALS
                   else "Activity (n.u.)")
supp_spectrum = BROADBAND_RAW_FIT * SUPP_UNIT_SCALE * BAND_RESPONSES[FIGURE7_BAND][:, None]
supp_trf = np.roll(np.fft.irfft(supp_spectrum, n=nperseg, axis=0), nperseg // 2, axis=0)
supp_envelope = np.abs(hilbert(supp_trf, axis=0))
supp_mean = supp_envelope.mean(-1)
supp_shown = (times >= SUPP_LAGS[0]) & (times <= SUPP_LAGS[1])

supp_limit = SUPP_YLIM
if supp_limit is None:
    supp_limit = float(supp_envelope[supp_shown].max()) * SUPP_HEADROOM

supp_fig = plt.figure(figsize=BROADBAND_FIGSIZE)
supp_grid = supp_fig.add_gridspec(2, 1, hspace=0.30)
supp_trf_ax = supp_fig.add_subplot(supp_grid[0])
supp_env_ax = supp_fig.add_subplot(supp_grid[1], sharex=supp_trf_ax)

for index in range(supp_trf.shape[1]):
    supp_trf_ax.plot(times[supp_shown], supp_trf[supp_shown, index],
                     color=CHANNEL_COLOURS[index], lw=0.5, alpha=0.55, zorder=2)
    supp_env_ax.plot(times[supp_shown], supp_envelope[supp_shown, index],
                     color=CHANNEL_COLOURS[index], lw=0.5, alpha=0.55, zorder=2)
supp_env_ax.plot(times[supp_shown], supp_mean[supp_shown], color=INK, lw=1.3, zorder=5)

for ax in (supp_trf_ax, supp_env_ax):
    ax.axvline(0, color=HAIRLINE, lw=0.8, zorder=0)
    ax.set_xlim(*SUPP_LAGS)
    ax.set_xticks(SUPP_XTICKS)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))
    ax.yaxis.get_offset_text().set_fontsize(SMALL_FONTSIZE)
    bare(ax)
supp_trf_ax.axhline(0, color=HAIRLINE, lw=0.8, zorder=0)
supp_trf_ax.set_ylim(-supp_limit, supp_limit)
supp_env_ax.set_ylim(0, supp_limit)
supp_trf_ax.set_ylabel(SUPP_UNIT_LABEL)
supp_env_ax.set_ylabel(SUPP_UNIT_LABEL)
supp_env_ax.set_xlabel("Lag (ms)")
supp_trf_ax.set_title(f"TRF ({FIGURE7_BAND[0]:g}-{FIGURE7_BAND[1]:g} Hz), no normalisation",
                      fontsize=DEFAULT_FONTSIZE)
supp_env_ax.set_title("Envelope of TRF", fontsize=DEFAULT_FONTSIZE)


def hang_on(fig, ax, data, lags, locations, width, cmap, symmetric,
            topo_w=None, topo_h=None):
    """`hang_topographies`, but taking the figure's axes explicitly.

    The version above closes over the axes it was defined against; calling it here would draw
    into figure 7.
    """
    for lag, (x, y) in zip(lags, locations):
        start = min(max(int(np.argmin(np.abs(times - lag))) - width // 2, 0),
                    len(times) - width)
        values = data[start:start + width].mean(0)
        ax.axvline(lag, color="k", lw=0.7, ls="--", alpha=0.75, zorder=1)
        topo_ax = ax.inset_axes([x, y, topo_w or TOPO_W, topo_h or TOPO_H])
        extreme = float(np.abs(values).max())
        plot_topomap(values, INFO, axes=topo_ax, show=False, cmap=cmap,
                     vlim=(-extreme, extreme) if symmetric else (0, extreme),
                     contours=0, sensors=False)
        topo_ax.set_title(f"{lag:g} ms", fontsize=SMALL_FONTSIZE, pad=2)


hang_on(supp_fig, supp_trf_ax, supp_trf, SUPP_TRF_TOPO_LAGS, SUPP_TRF_TOPO_LOCS,
        TRF_TOPO_WIDTH, "RdBu_r", symmetric=True)
hang_on(supp_fig, supp_env_ax, supp_envelope, SUPP_ENVELOPE_TOPO_LAGS,
        SUPP_ENVELOPE_TOPO_LOCS, ENVELOPE_TOPO_WIDTH, "viridis", symmetric=False)
plot_sensor_key(supp_trf_ax.inset_axes(BROADBAND_KEY_RECT))

for ax, label in [(supp_trf_ax, "a)"), (supp_env_ax, "b)")]:
    ax.text(-0.09, 1.02, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

stem = f"trf_broadband_{FIGURE7_BAND[0]:g}-{FIGURE7_BAND[1]:g}Hz_unnormalised_{REFERENCE}"
supp_fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
supp_fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

# --- the comparison this figure exists for -----------------------------------
centre = [INFO.ch_names.index(name) for name in CENTRE_SENSORS if name in INFO.ch_names]
rim = [INFO.ch_names.index(name) for name in RIM_SENSORS if name in INFO.ch_names]
print(f"{len(centre)} midline vs {len(rim)} rim sensors, envelope at each map's lag:")
print(f"{'lag':>9s}  {'divided by SD':>14s}  {'no normalisation':>18s}")
for lag in SUPP_ENVELOPE_TOPO_LAGS:
    start = min(max(int(np.argmin(np.abs(times - lag))) - ENVELOPE_TOPO_WIDTH // 2, 0),
                len(times) - ENVELOPE_TOPO_WIDTH)
    window = slice(start, start + ENVELOPE_TOPO_WIDTH)
    ratios = [values[window].mean(0)[centre].mean() / values[window].mean(0)[rim].mean()
              for values in (broadband_envelope, supp_envelope)]
    print(f"{lag:6.1f} ms  {ratios[0]:13.2f}x  {ratios[1]:17.2f}x")
print("centre/rim > 1 means the map is fronto-central; < 1 means it tracks EEG amplitude.")

# The shape claim, measured. Within a trial the divisor is a constant at each channel and cannot
# bend that channel's trace -- but it varies BETWEEN trials, so the pooled average is a
# re-weighted combination of them and is not a pure rescaling.
window_times = times[supp_shown]
correlations = np.array([
    np.corrcoef(broadband_envelope[supp_shown, i], supp_envelope[supp_shown, i])[0, 1]
    for i in range(supp_trf.shape[1])])
shifts = np.array([
    abs(window_times[np.argmax(broadband_envelope[supp_shown, i])]
        - window_times[np.argmax(supp_envelope[supp_shown, i])])
    for i in range(supp_trf.shape[1])])
print(f"\nper-channel envelope, normalised vs not: correlation median "
      f"{np.median(correlations):.4f} (min {correlations.min():.3f}); maximum in the same "
      f"sample for {int((shifts == 0).sum())} of {len(shifts)} channels, "
      f"largest shift {shifts.max():.1f} ms")

# The claims the text makes of this comparison, measured: the envelope's peak lags with and
# without the normalisation, and how the response divides between the hemispheres in each.
print("\nchannel-mean envelope local maxima, 0-60 ms: normalised "
      + ", ".join(f"{lag:.1f}" for lag in envelope_peaks(broadband_mean))
      + " ms; unnormalised " + ", ".join(f"{lag:.1f}" for lag in envelope_peaks(supp_mean)) + " ms")
side_of = {name: ("left" if int(name.rstrip("z")[-1]) % 2 else "right")
           for name in INFO.ch_names if name[-1].isdigit()}
left = [i for i, name in enumerate(INFO.ch_names) if side_of.get(name) == "left"]
right = [i for i, name in enumerate(INFO.ch_names) if side_of.get(name) == "right"]
print(f"right/left envelope ratio ({len(right)} right, {len(left)} left channels):")
print(f"{'window':>14s}  {'normalised':>11s}  {'unnormalised':>13s}")
for window in [(4.0, 10.0), (15.0, 30.0), (30.0, 50.0), (40.0, 44.0)]:
    rows = (times >= window[0]) & (times <= window[1])
    ratios = [values[rows][:, right].mean() / values[rows][:, left].mean()
              for values in (broadband_envelope, supp_envelope)]
    print(f"{f'{window[0]:g}-{window[1]:g} ms':>14s}  {ratios[0]:10.2f}x  {ratios[1]:12.2f}x")

## Figure 8 -- the spectrum cut in two

The transfer function over the cortical range and over everything above it, side by side. No
envelopes here: the point is the shape of the response itself, and at these two bandwidths the
shapes are different kinds of thing -- a slow deflection on the left, a fast oscillation on the
right.

Both are normalised, both use the same filter design and mains notches as the figures above, and
both are pooled in the same pass as everything else, so the two panels and figures 6 and 7 are
all the same trials under the same weights.

The lag windows differ per panel, since a 0.2-30 Hz response runs for hundreds of milliseconds
and a 30-300 Hz one is over within tens.

In [ ]:
# ===================== figure 8 options ======================================
SPLIT_FIGSIZE = (6., 4.4)
SPLIT_LAGS = [(-200.0, 800.0), (-50.0, 100.0)]
SPLIT_XTICKS = [[-200, 0, 200, 400, 600, 800], [-50, 0, 50, 100]]
SPLIT_YLIM = [None, None]                # None autoscales that panel
# Per panel, because the two are limited by different things. The maps sit with their feet at
# axes fraction 0.62, i.e. at 0.24 of the limit, so the traces beneath them have to stay under
# that. Measured, the largest trace under a map is 0.26 of the envelope maximum in (a) but only
# 0.10 in (b) -- so (a) needs ~1.1 and (b) is limited instead by not clipping its own peak,
# which reaches 0.98 of the envelope. These leave a little margin on each.
SPLIT_HEADROOM = [1.25, 1.05]

# Stacked, so each panel is full width and short -- the same shape as figure 7's, and the same
# rule applies: the HEIGHT sets the drawn size, the head sits at `x + SPLIT_TOPO_W / 2` and
# spans a little under +/-0.05 of the panel width about that centre.
SPLIT_TOPO_W, SPLIT_TOPO_H = 0.12, 0.30
SPLIT_TOPO_WIDTH = 3
SPLIT_TOPO_LAGS = [[43.0, 72.0, 101.0, 130.0],
                   [5.9, 8.8, 17.6, 23.4, 30.3, 42.0]]
# Split either side of the response rather than all to one side of it -- at six maps there is
# no longer room on one flank. (a)'s traces occupy x 0.20-0.55 of its panel and (b)'s
# x 0.33-0.62, so the earlier lags go left of that and the later ones right, which also keeps
# them ascending across the panel.
# The left edge (a)'s first map and the sensor key both sit on, so the two stay in a column
# whichever of them moves.
SPLIT_LEFT_X = 0.0
SPLIT_TOPO_LOCS = [[(SPLIT_LEFT_X, 0.62),
                    (0.57, 0.62), (0.69, 0.62), (0.81, 0.62)],
                   [(-0.01, 0.62), (0.085, 0.62), (0.18, 0.62),
                    (0.64, 0.62), (0.76, 0.62), (0.88, 0.62)]]
SPLIT_KEY_RECT = [SPLIT_LEFT_X, 0.04, SPLIT_TOPO_W, SPLIT_TOPO_H]
# =============================================================================

assert (len(SPLIT_LAGS) == len(SPLIT_XTICKS) == len(SPLIT_YLIM)
        == len(SPLIT_HEADROOM) == len(SPLIT_BANDS))
for lags, locations in zip(SPLIT_TOPO_LAGS, SPLIT_TOPO_LOCS):
    assert len(lags) == len(locations), "one position per lag in each panel"

# Stacked, and deliberately NOT sharing an x axis: the two bands are read over different
# spans, so each carries its own range, ticks and label.
split_fig, split_axes = plt.subplots(len(SPLIT_BANDS), 1, figsize=SPLIT_FIGSIZE)
split_fig.subplots_adjust(hspace=0.45)

for column, band in enumerate(SPLIT_BANDS):
    ax = split_axes[column]
    trf, envelope = band_response_in_lag(band)
    lags = SPLIT_LAGS[column]
    shown = (times >= lags[0]) & (times <= lags[1])

    for index in range(trf.shape[1]):
        ax.plot(times[shown], trf[shown, index], color=CHANNEL_COLOURS[index],
                lw=0.5, alpha=0.55, zorder=2)

    # The envelope is not drawn, but it still sets the scale: it bounds the trace it came from,
    # so a limit that fits the envelope cannot clip the traces.
    limit = SPLIT_YLIM[column]
    if limit is None:
        limit = float(envelope[shown].max()) * SPLIT_HEADROOM[column]
    ax.set_ylim(-limit, limit)
    ax.set_xlim(*lags)
    ax.set_xticks(SPLIT_XTICKS[column])
    ax.axvline(0, color=HAIRLINE, lw=0.8, zorder=0)
    ax.axhline(0, color=HAIRLINE, lw=0.8, zorder=0)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))
    ax.yaxis.get_offset_text().set_fontsize(SMALL_FONTSIZE)
    ax.set_xlabel("Lag (ms)")
    ax.set_title(f"{band[0]:g}-{band[1]:g} Hz", fontsize=DEFAULT_FONTSIZE)
    bare(ax)

    hang_on(split_fig, ax, trf, SPLIT_TOPO_LAGS[column], SPLIT_TOPO_LOCS[column],
            SPLIT_TOPO_WIDTH, "RdBu_r", symmetric=True,
            topo_w=SPLIT_TOPO_W, topo_h=SPLIT_TOPO_H)

for ax in split_axes:
    ax.set_ylabel("Activity (a.u.)")
plot_sensor_key(split_axes[0].inset_axes(SPLIT_KEY_RECT))

for ax, label in zip(split_axes, ("a)", "b)")):
    ax.text(-0.09, 1.03, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

stem = f"trf_split_{REFERENCE}"
split_fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
split_fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

for column, band in enumerate(SPLIT_BANDS):
    trf, envelope = band_response_in_lag(band)
    lags = SPLIT_LAGS[column]
    shown = (times >= lags[0]) & (times <= lags[1])
    peak = int(np.argmax(np.where(shown, envelope.mean(-1), -np.inf)))
    used, wanted, transition, worst, where, dc = BAND_QUALITY[band]
    print(f"{f'{band[0]:g}-{band[1]:g} Hz':>14s}  envelope peak {times[peak]:6.1f} ms "
          f"(strongest {INFO.ch_names[int(np.argmax(envelope[peak]))]:>4s})  "
          f"filter {transition:g} Hz, stopband {worst:+.0f} dB, DC {dc:+.0f} dB"
          + ("   <- DC in the passband" if LEAKY[band] else ""))

### Figure 8, alternative layout

The same two bands, but side by side with their topographies in a grid underneath rather than
hung inside the plot area. The trade is straightforward: nothing overlaps the traces, so the
y limits can be pulled in and the response drawn at full size, but the maps no longer sit next
to the lag they belong to and have to be tied to it by the rules and their titles instead.

Each grid is two rows, as many columns as that panel's lags need. Everything else -- bands,
filters, pooling, lags -- is figure 8's, read from the same constants, so the two layouts cannot
drift apart.

In [ ]:
# ================= alternative layout options ================================
ALT_FIGSIZE = (6.27, 4.6)
# Nothing hangs in the plot area now, so the traces can fill it.
ALT_HEADROOM = [1.05, 1.02]
ALT_TOPO_ROWS = 2
ALT_HSPACE, ALT_WSPACE = 0.45, 0.30      # between the trace row and the grids, and columns
ALT_GRID_WSPACE, ALT_GRID_HSPACE = 0.05, 0.45
ALT_KEY_RECT = [0.0, 0.66, 0.16, 0.32]   # sensor key, in the left trace panel
ALT_HEIGHT_RATIOS = [2.1, 1.9]           # trace row : topography rows

# Extra maps for THIS layout only, one list per panel, appended after figure 8's own lags. A
# `(low, high)` entry is averaged over that whole lag range rather than over a few samples at a
# point -- the late response is broad and low, so a single sample of it is mostly noise. Figure
# 8 hangs its maps in the plot area and has no room for more; a grid does.
ALT_EXTRA_LAGS = [[(220.0, 450.0)], []]
ALT_SPAN_ALPHA = 0.10                    # the shading that marks a range on the traces
# =============================================================================

assert len(ALT_HEADROOM) == len(ALT_EXTRA_LAGS) == len(SPLIT_BANDS), "one entry per band"


def alt_window(entry):
    """`(slice, label)` for a single lag or for a `(low, high)` range averaged over.

    A point is `SPLIT_TOPO_WIDTH` samples centred on it, clipped at the ends of the transform
    rather than wrapped -- the lag axis is circular with the window's period, so a window
    running off the edge would come back a whole period from where it claims to be.
    """
    if np.isscalar(entry):
        start = min(max(int(np.argmin(np.abs(times - entry))) - SPLIT_TOPO_WIDTH // 2, 0),
                    len(times) - SPLIT_TOPO_WIDTH)
        return slice(start, start + SPLIT_TOPO_WIDTH), f"{entry:g} ms"
    low, high = entry
    inside = np.flatnonzero((times >= low) & (times <= high))
    return slice(inside[0], inside[-1] + 1), f"{low:g}-{high:g} ms"

alt_fig = plt.figure(figsize=ALT_FIGSIZE)
alt_outer = alt_fig.add_gridspec(2, len(SPLIT_BANDS), height_ratios=ALT_HEIGHT_RATIOS,
                                 hspace=ALT_HSPACE, wspace=ALT_WSPACE)

for column, band in enumerate(SPLIT_BANDS):
    trf, envelope = band_response_in_lag(band)
    lags = SPLIT_LAGS[column]
    shown = (times >= lags[0]) & (times <= lags[1])
    topo_lags = list(SPLIT_TOPO_LAGS[column]) + list(ALT_EXTRA_LAGS[column])

    # --- the traces ----------------------------------------------------------
    ax = alt_fig.add_subplot(alt_outer[0, column])
    for index in range(trf.shape[1]):
        ax.plot(times[shown], trf[shown, index], color=CHANNEL_COLOURS[index],
                lw=0.5, alpha=0.55, zorder=2)
    for entry in topo_lags:
        if np.isscalar(entry):
            ax.axvline(entry, color="k", lw=0.7, ls="--", alpha=0.75, zorder=1)
        else:
            # A band of shading, since what the map averages is a stretch and not an instant.
            ax.axvspan(*entry, color="k", alpha=ALT_SPAN_ALPHA, lw=0, zorder=0)

    # The envelope bounds the trace it came from, so a limit that fits it cannot clip.
    limit = float(envelope[shown].max()) * ALT_HEADROOM[column]
    ax.set_ylim(-limit, limit)
    ax.set_xlim(*lags)
    ax.set_xticks(SPLIT_XTICKS[column])
    ax.axvline(0, color=HAIRLINE, lw=0.8, zorder=0)
    ax.axhline(0, color=HAIRLINE, lw=0.8, zorder=0)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))
    ax.yaxis.get_offset_text().set_fontsize(SMALL_FONTSIZE)
    ax.set_xlabel("Lag (ms)")
    ax.set_ylabel("Activity (a.u.)")
    ax.set_title(f"{band[0]:g}-{band[1]:g} Hz", fontsize=DEFAULT_FONTSIZE)
    bare(ax)
    ax.text(-0.26, 1.06, f"{'ab'[column]})", transform=ax.transAxes,
            fontsize=PANEL_LABEL_FONTSIZE, fontweight="bold", va="bottom", ha="left")
    if column == 0:
        plot_sensor_key(ax.inset_axes(ALT_KEY_RECT))

    # --- the grid of maps ----------------------------------------------------
    # Two rows, and as many columns as this panel's lags need. With the row count fixed the
    # cell height is the same in both grids, and `plot_topomap` keeps the head square, so the
    # heads come out nearly the same size on both sides despite the different column counts.
    columns = -(-len(topo_lags) // ALT_TOPO_ROWS)
    grid = alt_outer[1, column].subgridspec(ALT_TOPO_ROWS, columns,
                                            wspace=ALT_GRID_WSPACE, hspace=ALT_GRID_HSPACE)
    for position, entry in enumerate(topo_lags):
        window, label = alt_window(entry)
        values = trf[window].mean(0)
        extreme = float(np.abs(values).max())
        topo_ax = alt_fig.add_subplot(grid[position // columns, position % columns])
        plot_topomap(values, INFO, axes=topo_ax, show=False, cmap="RdBu_r",
                     vlim=(-extreme, extreme), contours=0, sensors=False)
        topo_ax.set_title(label, fontsize=SMALL_FONTSIZE, pad=2)

stem = f"trf_split_grid_{REFERENCE}"
alt_fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
alt_fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

for column, band in enumerate(SPLIT_BANDS):
    lags = list(SPLIT_TOPO_LAGS[column]) + list(ALT_EXTRA_LAGS[column])
    print(f"{f'{band[0]:g}-{band[1]:g} Hz':>14s}: {len(lags)} maps in "
          f"{ALT_TOPO_ROWS}x{-(-len(lags) // ALT_TOPO_ROWS)}  "
          + ", ".join(alt_window(entry)[1] for entry in lags))

## Figure 9 -- latency against centre frequency

A bank of narrow bands across the gamma range, the lag at which each one's envelope peaks, and
that latency against the band's centre. The envelope rather than the transfer function itself,
because a narrowband response is an oscillation whose carrier phase is arbitrary -- its own peaks
move with the band while the envelope's do not, and since the filter is zero phase the envelope's
peak is an unbiased estimate of the lag.

Its *precision* is set by the bandwidth. A band of width $w$ spreads a feature over roughly
$1/w$, so a 10 Hz band localises a 7 ms response only to within tens of milliseconds. That is
what the scatter about the plateau in (a) is; it is not disagreement between bands.

The description is piecewise -- a linear fall over the lower half, a constant over the upper --
fitted to bands centred up to 135 Hz and from 140 Hz respectively. Both limits are set as
**frequencies** rather than as indices into the band list, so they survive a change to the
grid.

These bands reuse the pooled broadband fit rather than being pooled again with a divisor each.
The normalisation equalises channels, and here every band is read through the same channel
average, so one divisor for all of them keeps the bands comparable with each other. The
per-band alternative is in `exploration/transfer_functions.ipynb` section 8 and costs a pass
over the trial files.

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# ===================== figure 9 options ======================================
NARROW_CENTRES = np.arange(30.0, 301.0, 5.0)
NARROW_WIDTH = 10.0             # a number, or an array the same length as the centres
# The narrow bands' filter transition, in place of the 1 Hz the band rule gives a 10 Hz band.
# Wider transitions mean a shorter kernel and gentler band edges, so less ringing in lag; the
# -6 dB edges stay at the band's limits, and each transition spans +/- half this about them.
NARROW_TRANSITION_HZ = 1.0

# Where the envelope peak is looked for. It has to start BELOW zero: a band wide enough to
# smear a 7 ms response over ~100 ms puts some estimates slightly negative, and a window
# starting at 0 turns those into a pile of latencies at exactly 0 -- an artefact of the window.
NARROW_SEARCH_LAGS = (-20.0, 250.0)
# What (b) draws, which is not the window the peak is searched in: the search has to be wide
# enough for the slowest band, and at that width the fast ones are mostly decaying tail.
NARROW_DISPLAY_LAGS = (-50.0, 100.0)

# The two descriptions are fitted over separate ranges of centre frequency.
LINEAR_MAX_HZ = 135.0           # the line is fitted to bands centred at or below this
PLATEAU_MIN_HZ = 145.0          # the constant to bands centred at or above this
# Bands flatter than this are dropped from both fits: peak over mean across the search window,
# so 1 is perfectly flat and the maximum is picking between comparable bumps.
MIN_PEAK_SHARPNESS = 1.15

NARROW_FIGSIZE = (6.27, 2.8)
NARROW_CMAP = "plasma"
NARROW_ALPHA = 0.65             # the traces overlap heavily; this lets the pile be seen through
# Both descriptions in the same dash, so the two are told apart by colour rather than by a
# second cue doing the same job. Vibrant against the black crosses and clear of the plasma
# ramp next door.
FIT_COLOUR, PLATEAU_COLOUR = "#e2431f", "#1178b0"
NARROW_XTICKS = [50, 100, 150, 200, 250, 300]
NARROW_DISPLAY_XTICKS = [-50, 0, 50, 100]
# =============================================================================

narrow_widths = np.broadcast_to(np.asarray(NARROW_WIDTH, float), NARROW_CENTRES.shape)
NARROW_BANDS = [(centre - width / 2, centre + width / 2)
                for centre, width in zip(NARROW_CENTRES, narrow_widths)]
assert all(low > 0 for low, _ in NARROW_BANDS), "a band reaches below DC"

# One filter per band, and the same leak check the five-band table runs.
narrow_responses, narrow_leaky = [], []
for low, high in NARROW_BANDS:
    response, used, wanted, transition = band_filter(low, high, NARROW_TRANSITION_HZ)
    narrow_responses.append(response * NOTCHES)
    narrow_leaky.append(filter_quality(response, low, high, transition)[0] > STOPBAND_WARN_DB)
narrow_leaky = np.array(narrow_leaky)

# Every band is a different mask on the same pooled fit, so this costs no extra pass.
narrow_fit = BAND_FITS[BROADBAND]
search = (times >= NARROW_SEARCH_LAGS[0]) & (times <= NARROW_SEARCH_LAGS[1])
shown = (times >= NARROW_DISPLAY_LAGS[0]) & (times <= NARROW_DISPLAY_LAGS[1])

narrow_envelopes, latencies, sharpness = [], [], []
for response in narrow_responses:
    spectrum = narrow_fit * response[:, None]
    trf = np.roll(np.fft.irfft(spectrum, n=nperseg, axis=0), nperseg // 2, axis=0)
    envelope = np.abs(hilbert(trf, axis=0)).mean(-1)
    peak = int(np.argmax(np.where(search, envelope, -np.inf)))
    narrow_envelopes.append(envelope)
    latencies.append(times[peak])
    sharpness.append(envelope[peak] / envelope[search].mean())

latencies = np.array(latencies)
sharpness = np.array(sharpness)
flat = ~narrow_leaky & (sharpness < MIN_PEAK_SHARPNESS)
usable = ~narrow_leaky & ~flat

# Envelopes first: they are what the latencies are measured FROM, so they read left to right
# as the argument does. The colourbar then sits in the gutter, which needs to be wider.
narrow_fig, (ax_envelopes, ax_latency) = plt.subplots(1, 2, figsize=NARROW_FIGSIZE)
narrow_fig.subplots_adjust(wspace=0.52)

# --- (a) latency against centre frequency ------------------------------------
ax_latency.scatter(NARROW_CENTRES[usable], latencies[usable], marker="x", s=14, c=INK,
                   linewidths=0.9, zorder=3)
if (~usable).any():
    ax_latency.scatter(NARROW_CENTRES[~usable], latencies[~usable], marker="x", s=14,
                       c=HAIRLINE, linewidths=0.9, zorder=2, label="excluded")

low_side = usable & (NARROW_CENTRES <= LINEAR_MAX_HZ)
high_side = usable & (NARROW_CENTRES >= PLATEAU_MIN_HZ)
gradient = plateau = None
if low_side.sum() >= 2:
    gradient, intercept = np.polyfit(NARROW_CENTRES[low_side], latencies[low_side], 1)
    span = np.linspace(NARROW_CENTRES[low_side].min(), NARROW_CENTRES[low_side].max(), 100)
    ax_latency.plot(span, gradient * span + intercept, color=FIT_COLOUR, ls="--", lw=1.6,
                    label=f"linear fit gradient: {gradient:.3f} ms/Hz", zorder=4)
if high_side.sum() >= 2:
    # Median rather than mean: the plateau is what is being estimated, and one band whose peak
    # landed on a sidelobe should not move it.
    plateau = float(np.median(latencies[high_side]))
    ax_latency.plot([NARROW_CENTRES[high_side].min(), NARROW_CENTRES[high_side].max()],
                    [plateau] * 2, color=PLATEAU_COLOUR, ls="--", lw=1.6,
                    label=f"Stable latency: {plateau:.1f} ms", zorder=4)

ax_latency.set_xlabel("EEG centre frequency (Hz)")
ax_latency.set_ylabel("Latency of envelope peak (ms)")
ax_latency.set_xticks(NARROW_XTICKS)
ax_latency.legend(fontsize=SMALL_FONTSIZE, frameon=False, loc="lower left")
bare(ax_latency)

# --- (b) every band's envelope, coloured by centre ---------------------------
# Each trace divided by its own maximum over what is drawn: the point is WHERE a band peaks,
# and the bands differ in amplitude across this range. Unlike standardising, this keeps the
# floor where it is, so a band whose envelope barely modulates stays visibly flat instead of
# being stretched to fill the panel like the rest.
norm = Normalize(vmin=0, vmax=len(NARROW_CENTRES) - 1)
colormap = plt.get_cmap(NARROW_CMAP)
for index, envelope in enumerate(narrow_envelopes):
    trace = envelope[shown]
    ax_envelopes.plot(times[shown], 100 * trace / trace.max(), color=colormap(norm(index)),
                      lw=0.8, alpha=NARROW_ALPHA)
ax_envelopes.set_ylim(0, 104)
ax_envelopes.set_yticks([0, 20, 40, 60, 80, 100])
ax_envelopes.axvline(0, color=HAIRLINE, lw=0.8, zorder=0)
ax_envelopes.set_xlim(*NARROW_DISPLAY_LAGS)
ax_envelopes.set_xticks(NARROW_DISPLAY_XTICKS)
ax_envelopes.set_xlabel("Lag (ms)")
ax_envelopes.set_ylabel("TRF envelope (% of max)")
bare(ax_envelopes)

# The colour axis is the band INDEX, so the bar is relabelled in Hz by interpolating the two
# against each other -- which is what keeps it right under a non-uniform grid.
mappable = ScalarMappable(norm=norm, cmap=colormap)
mappable.set_array([])
colorbar = plt.colorbar(mappable, ax=ax_envelopes, pad=0.03)
colorbar.set_ticks(np.interp(NARROW_XTICKS, NARROW_CENTRES, np.arange(len(NARROW_CENTRES))),
                   labels=[f"{value:g}" for value in NARROW_XTICKS])
colorbar.set_label("Centre frequency (Hz)", fontsize=SMALL_FONTSIZE)
colorbar.ax.tick_params(labelsize=TICK_FONTSIZE)

for ax, label in [(ax_envelopes, "a)"), (ax_latency, "b)")]:
    ax.text(-0.24, 1.03, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

stem = f"trf_narrowband_latency_{REFERENCE}"
narrow_fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
narrow_fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

print(f"{len(NARROW_CENTRES)} bands, centres {NARROW_CENTRES[0]:g}-{NARROW_CENTRES[-1]:g} Hz, "
      f"{narrow_widths[0]:g} Hz wide; {int(usable.sum())} usable "
      f"({int(narrow_leaky.sum())} leaky, {int(flat.sum())} too flat)")
if np.isin(latencies, NARROW_SEARCH_LAGS).any():
    print(f"  {int(np.isin(latencies, NARROW_SEARCH_LAGS).sum())} peaks landed on an edge of "
          f"the search window -- widen NARROW_SEARCH_LAGS")
if gradient is not None:
    print(f"at or below {LINEAR_MAX_HZ:g} Hz: {gradient:+.4f} ms/Hz over {int(low_side.sum())} "
          f"bands ({latencies[low_side].max():.1f} down to {latencies[low_side].min():.1f} ms)")
if plateau is not None:
    spread = latencies[high_side]
    print(f"at or above {PLATEAU_MIN_HZ:g} Hz: median {plateau:.1f} ms, range "
          f"{spread.min():.1f}-{spread.max():.1f} ms over {int(high_side.sum())} bands")

## Phase-only transfer function

Figure 7 again, but with the magnitude of the transfer function thrown away. The pooled fit is
divided by its own magnitude, bin by bin and channel by channel, so every frequency enters with
unit gain and only its phase is kept:

$$H_\phi(f) = \frac{H(f)}{|H(f)|}$$

After that it is treated exactly as figure 7 treats $H$: the same band filter and mains notches,
the same inverse transform, the same analytic envelope. Since only phase is left, what the
traces show is timing alone: a lag at which the phases of many frequencies agree comes out as a
peak, however little power those frequencies carry. This is the same idea as the PHAT weighting
used in time-delay estimation.

Two things follow from that:

- **The per-channel SD divisor drops out.** It is one real, positive number per channel, so
  dividing by the magnitude removes it. The normalised and undivided fits give the same
  phase-only transfer function.
- **Noise bins count as much as signal bins.** A frequency where the EEG does not track the
  envelope contributes a random phase at full weight, where in $H$ it would have contributed
  almost nothing. Above roughly 200 Hz the coherence is at its noise level, so a band that
  reaches up there adds a floor of incoherent ripple at every lag. `PHASE_BAND` picks the band
  the phase is read through. It can be any band that is pooled above (`ALL_BANDS`).

The phase is taken from the pooled $H$, not per trial before pooling. The pooled phase is the
phase of the duration-weighted average, so trials whose phases disagree still cancel there
before the magnitude is removed.

In [ ]:
# ===================== phase-only options ====================================
PHASE_BAND = BROADBAND          # any band in ALL_BANDS: those are the fits pooled above
PHASE_LAGS = (-50.0, 500.0)
PHASE_XTICKS = [-50, 0, 50, 100]
PHASE_HEADROOM = 1.15
PHASE_FIGSIZE = (6.27, 4.2)
PHASE_KEY_RECT = [0.0, 0.62, 0.11, 0.36]   # sensor key, in the top panel
# =============================================================================

if PHASE_BAND not in BAND_FITS:
    raise ValueError(f"{PHASE_BAND} is not pooled; pick one of {list(BAND_FITS)}")

# Unit magnitude in every bin with any fit at all. A bin with |H| exactly 0 has no phase to
# keep, so it stays 0 rather than becoming a division by zero.
fit = BAND_FITS[PHASE_BAND]
magnitude = np.abs(fit)
phase_spectrum = np.divide(fit, magnitude, out=np.zeros_like(fit), where=magnitude > 0)

# From here, `band_response_in_lag`'s two lines on the phase-only spectrum.
phase_trf = np.roll(np.fft.irfft(phase_spectrum * BAND_RESPONSES[PHASE_BAND][:, None],
                                 n=nperseg, axis=0), nperseg // 2, axis=0)
phase_envelope = np.abs(hilbert(phase_trf, axis=0))
phase_mean = phase_envelope.mean(-1)
phase_shown = (times >= PHASE_LAGS[0]) & (times <= PHASE_LAGS[1])
phase_limit = float(phase_envelope[phase_shown].max()) * PHASE_HEADROOM

phase_fig = plt.figure(figsize=PHASE_FIGSIZE)
phase_grid = phase_fig.add_gridspec(2, 1, hspace=0.30)
ax_phase_trf = phase_fig.add_subplot(phase_grid[0])
ax_phase_env = phase_fig.add_subplot(phase_grid[1], sharex=ax_phase_trf)

for index in range(phase_trf.shape[1]):
    ax_phase_trf.plot(times[phase_shown], phase_trf[phase_shown, index],
                      color=CHANNEL_COLOURS[index], lw=0.5, alpha=0.55, zorder=2)
    ax_phase_env.plot(times[phase_shown], phase_envelope[phase_shown, index],
                      color=CHANNEL_COLOURS[index], lw=0.5, alpha=0.55, zorder=2)
ax_phase_env.plot(times[phase_shown], phase_mean[phase_shown], color=INK, lw=1.3, zorder=5)

for ax in (ax_phase_trf, ax_phase_env):
    ax.axvline(0, color=HAIRLINE, lw=0.8, zorder=0)
    ax.set_xlim(*PHASE_LAGS)
    ax.set_xticks(PHASE_XTICKS)
    ax.set_ylabel("Activity (a.u.)")
    bare(ax)
ax_phase_trf.axhline(0, color=HAIRLINE, lw=0.8, zorder=0)
ax_phase_trf.set_ylim(-phase_limit, phase_limit)
ax_phase_env.set_ylim(0, phase_limit)
ax_phase_env.set_xlabel("Lag (ms)")
# Shared lag axis: labelled once, at the bottom, so b)'s letter has the gap to itself.
ax_phase_trf.tick_params(axis="x", labelbottom=False)
ax_phase_trf.set_title(f"Phase-only TRF ({PHASE_BAND[0]:g}-{PHASE_BAND[1]:g} Hz)",
                       fontsize=DEFAULT_FONTSIZE)
ax_phase_env.set_title("Envelope of phase-only TRF", fontsize=DEFAULT_FONTSIZE)
plot_sensor_key(ax_phase_trf.inset_axes(PHASE_KEY_RECT))

for ax, label in [(ax_phase_trf, "a)"), (ax_phase_env, "b)")]:
    ax.text(-0.09, 1.02, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

stem = f"trf_phase_only_{PHASE_BAND[0]:g}-{PHASE_BAND[1]:g}Hz_{REFERENCE}"
phase_fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
phase_fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

# Against the ordinary fit read through the same band, for the caption.
ordinary_mean = band_response_in_lag(PHASE_BAND)[1].mean(-1)
phase_peak = int(np.argmax(np.where(phase_shown, phase_mean, -np.inf)))
ordinary_peak = int(np.argmax(np.where(phase_shown, ordinary_mean, -np.inf)))
outside = (times < -200) | (times > 500)
print(f"{PHASE_BAND[0]:g}-{PHASE_BAND[1]:g} Hz, {n_subjects} participants")
print(f"  phase-only channel-mean envelope peaks at {times[phase_peak]:.1f} ms "
      f"(strongest {INFO.ch_names[int(np.argmax(phase_envelope[phase_peak]))]}); "
      f"ordinary fit at {times[ordinary_peak]:.1f} ms")
print(f"  peak over the median envelope away from the response (|lag| outside -200..500 ms): "
      f"phase-only {phase_mean[phase_peak] / np.median(phase_mean[outside]):.1f}x, "
      f"ordinary {ordinary_mean[ordinary_peak] / np.median(ordinary_mean[outside]):.1f}x")

## Two reweightings of the broadband transfer function

Figure 7's broadband TRF is dominated by the response above 30 Hz. That is not because $|H|$ is
larger there: per bin it is comparable to delta. It is because a TRF's peak is a sum over bins,
and 30-300 Hz has about nine times as many 0.2 Hz bins as 0.2-30 Hz. On top of that, the
brainstem and midlatency response is close to a pure delay, so those bins add up in phase. Two
reweightings of $H$ before the inverse transform redistribute that, each answering a different
question. Neither touches the fit. They are display weightings, applied at the point of
transforming.

- **(a) $H(f)/f$: equal weight per octave.** Each octave holds a number of bins proportional to its
  centre frequency, so dividing by $f$ lets every octave contribute equally rather than every
  hertz. This is the time-domain counterpart of the $f\,P(f)$ argument for the modulation
  spectra. The DC bin has no octave and is set to 0.
- **(b) $H(f)\sqrt{S_{xx}(f)}$: the response to a regressor standardised at each frequency.** This
  equals $C_{xy}\sqrt{S_{yy}}$ with $H$'s phase, so the magnitude it draws is how much of the
  EEG at each frequency the envelope predicts. It undoes the $1/S_{xx}$ boost that a weak
  high-frequency regressor gives $H$. $S_{xx}$ is pooled from the same trial files with the same
  participant weights as $H$, and cached.

Both are on arbitrary units, and the two panels are not on a common scale.

Both weightings tilt the spectrum towards its lowest bins, so the lower edge of the band they are
read through matters. At 0.2 Hz the panels draw per-channel offsets and drift, because DC is
rejected by only 9 dB in a 5 s window. `WEIGHTED_BAND` therefore starts at 1 Hz by default.

In [ ]:
# ===================== reweighted-TRF options ================================
# The band the reweighted fit is read through. Its lower edge matters far more here than in
# figure 7: both weightings tilt the spectrum towards the bottom bins, and at 0.2 Hz the 5 s
# window rejects DC by only 9 dB (the band table above), so a 0.2 Hz edge draws per-channel
# offsets and drift rather than a response. 1 Hz rejects DC properly. Any (low, high) works --
# the filter is designed here if the band is not one pooled above.
WEIGHTED_BAND = (30.0, 250.0)
WEIGHTED_LAGS = (-100.0, 100.0)      # wide enough for the cortical response, which is the point
WEIGHTED_XTICKS = [-100, 0, 100]#, 200, 300, 400, 500]
WEIGHTED_HEADROOM = 1.1
WEIGHTED_FIGSIZE = (6.27, 7.0)
WEIGHTED_HSPACE = 0.55
# Where the maps are taken: the components the traces show. A number is a lag, averaged over
# WEIGHTED_TOPO_WIDTH samples about it; a (low, high) pair is averaged over that whole range.
WEIGHTED_TOPO_LAGS = [7.8, 23.4, 43.0, 100.0, 150.0, (250.0, 450.0)]
WEIGHTED_TOPO_WIDTH = 3
WEIGHTED_SPAN_ALPHA = 0.08
WEIGHTED_KEY_RECT = [0.88, 0.60, 0.11, 0.38]   # sensor key, in the top panel
# True reads figure 7's fit (per-channel EEG SD divided out); False the undivided one. For one
# band the divisor is one number per channel, so this changes the channels' relative scale only.
WEIGHTED_NORMALISED = True
# =============================================================================


def stimulus_psd_cache_path():
    key = json.dumps([REFERENCE, WINDOW_SECONDS, OVERLAP, FEATURE, len(paths), 1]
                     + ([["pascals", PRESENTATION_LEVEL_DBA, CALIBRATION_TUBE_FILTER,
                          ENVELOPE_COMPRESSION]] if ENVELOPE_IN_PASCALS else []),
                     sort_keys=True)
    return CACHE_ROOT / REFERENCE / f"trf_stimulus_psd_{hashlib.sha1(key.encode()).hexdigest()[:12]}.npz"


def pool_stimulus_psd():
    """S_xx pooled with H's own weights: duration within a participant, then equal participants.

    Read key by key from each npz rather than through `load_trial`, which would also pull in
    the (F, 66) cross-spectrum and PSDs that this does not need.
    """
    entries = []
    for path in paths:
        with np.load(path) as saved:
            psd = saved[f"stimulus_psd_{FEATURE}"].astype(float)
            # In the same units as H: with the envelope in Pa^0.6 it scales by k^0.6, so its
            # power by k^1.2. Otherwise H * sqrt(S_xx) would mix Pa^0.6 with digital units.
            if ENVELOPE_IN_PASCALS:
                psd = psd * ENVELOPE_FACTOR[TRIAL_STIMULUS[path]] ** 2
            entries.append((str(saved["subject"]), float(saved["analysed_duration"]), psd))
    totals = {}
    for subject, duration, _ in entries:
        totals[subject] = totals.get(subject, 0.0) + duration
    pooled = np.zeros_like(entries[0][2])
    for subject, duration, psd in entries:
        pooled += psd * duration / totals[subject] / len(totals)
    return pooled


psd_cache = stimulus_psd_cache_path()
if psd_cache.exists():
    with np.load(psd_cache) as saved:
        STIMULUS_PSD = saved["stimulus_psd"]
else:
    STIMULUS_PSD = pool_stimulus_psd()
    np.savez(psd_cache, stimulus_psd=STIMULUS_PSD)

# The broadband fit either way. Its SD divisor is one number per channel, so reading it through
# another band is still a legitimate fit -- only the per-band SD of figure 6 would differ.
weighted_fit = BAND_FITS[BROADBAND] if WEIGHTED_NORMALISED else BROADBAND_RAW_FIT
weighted_response = (BAND_RESPONSES[WEIGHTED_BAND] if WEIGHTED_BAND in BAND_RESPONSES
                     else band_filter(*WEIGHTED_BAND)[0] * NOTCHES)

inverse_frequency = np.divide(1.0, frequencies, out=np.zeros_like(frequencies),
                              where=frequencies > 0)
WEIGHTINGS = {
    "per_octave": (inverse_frequency, r"$H(f)\,/\,f$ (equal weight per octave)"),
    "standardised": (np.sqrt(STIMULUS_PSD),
                     r"$H(f)\,\sqrt{S_{xx}(f)}$ (regressor standardised per frequency)"),
}


def weighted_trf(weight):
    """`band_response_in_lag`'s two lines, with `weight` multiplied onto H first."""
    spectrum = weighted_fit * (weight * weighted_response)[:, None]
    return np.roll(np.fft.irfft(spectrum, n=nperseg, axis=0), nperseg // 2, axis=0)


weighted_shown = (times >= WEIGHTED_LAGS[0]) & (times <= WEIGHTED_LAGS[1])


def weighted_window(entry):
    """`(slice, label)`: a few samples about a lag, or a whole `(low, high)` range averaged.

    A range for the late slow wave, which is broad and low, so a single sample of it is mostly
    noise. Clipped at the ends of the transform rather than wrapped: the lag axis is circular.
    """
    if np.isscalar(entry):
        centre = int(np.argmin(np.abs(times - entry)))
        start = min(max(centre - WEIGHTED_TOPO_WIDTH // 2, 0), len(times) - WEIGHTED_TOPO_WIDTH)
        return slice(start, start + WEIGHTED_TOPO_WIDTH), f"{entry:g} ms"
    low, high = entry
    indices = np.flatnonzero((times >= low) & (times <= high))
    return slice(indices[0], indices[-1] + 1), f"{low:g}-{high:g} ms"


weighted_fig = plt.figure(figsize=WEIGHTED_FIGSIZE)
# Trace, then its maps, twice. The strip sits under its own panel so a map is read against the
# trace it was taken from.
weighted_grid = weighted_fig.add_gridspec(4, 1, height_ratios=[3, 1.25, 3, 1.25],
                                          hspace=WEIGHTED_HSPACE)
weighted_axes = [weighted_fig.add_subplot(weighted_grid[0])]
weighted_axes.append(weighted_fig.add_subplot(weighted_grid[2], sharex=weighted_axes[0]))
weighted_trfs = {}
for row, (ax, (name, (weight, title))) in enumerate(zip(weighted_axes, WEIGHTINGS.items())):
    trf = weighted_trfs[name] = weighted_trf(weight)
    for index in range(trf.shape[1]):
        ax.plot(times[weighted_shown], trf[weighted_shown, index],
                color=CHANNEL_COLOURS[index], lw=0.5, alpha=0.55, zorder=2)
    limit = float(np.abs(trf[weighted_shown]).max()) * WEIGHTED_HEADROOM
    ax.set_ylim(-limit, limit)
    ax.axhline(0, color=HAIRLINE, lw=0.8, zorder=0)
    ax.axvline(0, color=HAIRLINE, lw=0.8, zorder=0)
    ax.set_xlim(*WEIGHTED_LAGS)
    ax.set_xticks(WEIGHTED_XTICKS)
    ax.set_ylabel("Activity (a.u.)")
    ax.set_xlabel("Lag (ms)")
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))
    ax.yaxis.get_offset_text().set_fontsize(SMALL_FONTSIZE)
    ax.set_title(title, fontsize=DEFAULT_FONTSIZE)
    bare(ax)

    # The maps: a dashed rule at a lag, a shaded span for a range, each on its own symmetric
    # scale -- the components differ in size by several times, and a shared scale would leave
    # the small ones blank.
    strip = weighted_grid[2 * row + 1].subgridspec(1, len(WEIGHTED_TOPO_LAGS), wspace=0.08)
    for column, entry in enumerate(WEIGHTED_TOPO_LAGS):
        window, label = weighted_window(entry)
        if np.isscalar(entry):
            ax.axvline(entry, color="k", lw=0.7, ls="--", alpha=0.7, zorder=1)
        else:
            ax.axvspan(*entry, color=INK, alpha=WEIGHTED_SPAN_ALPHA, lw=0, zorder=0)
        values = trf[window].mean(0)
        extreme = float(np.abs(values).max())
        topo_ax = weighted_fig.add_subplot(strip[column])
        plot_topomap(values, INFO, axes=topo_ax, show=False, cmap="RdBu_r",
                     vlim=(-extreme, extreme), contours=0, sensors=False)
        topo_ax.set_title(label, fontsize=SMALL_FONTSIZE, pad=2)

plot_sensor_key(weighted_axes[0].inset_axes(WEIGHTED_KEY_RECT))

for ax, label in zip(weighted_axes, ["a)", "b)"]):
    ax.text(-0.09, 1.02, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

stem = (f"trf_reweighted_{WEIGHTED_BAND[0]:g}-{WEIGHTED_BAND[1]:g}Hz_{REFERENCE}"
        f"{'' if WEIGHTED_NORMALISED else '_unnormalised'}")
weighted_fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
weighted_fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

# How much of each weighted spectrum's energy sits below 30 Hz, against the plain fit -- the
# number the panels are meant to change. Energy, by Parseval, so it does not depend on lag.
in_band = weighted_response ** 2
low = frequencies < 30.0
for name, weight in [("plain H", np.ones_like(frequencies)),
                     *[(name, weight) for name, (weight, _) in WEIGHTINGS.items()]]:
    energy = (np.abs(weighted_fit) ** 2).mean(-1) * weight ** 2 * in_band
    print(f"{name:>13s}: {100 * energy[low].sum() / energy.sum():5.1f}% of the energy below 30 Hz")

## Figure 10 -- the per-frequency standardised TRF

$H(f)\sqrt{S_{xx}(f)}$: the response to a regressor standardised at each frequency (reweighting
(b) above), over three bands.

* **(a)** 1-30 Hz.
* **(b)** 30-250 Hz.
* **(c)** 1-250 Hz, both together.

Each panel also shows, in black under the butterfly, the mean over channels of each channel's
analytic envelope. It uses the same units and scale, with its zero (the grey hairline) moved below
the traces.

The EEG normalisation is the per-band one of figures 6 and 8. Each panel's fit is divided, per
channel and per trial, by the SD of the EEG within that panel's own band. $S_{xx}$ is the
envelope power pooled with $H$'s weights, in the same units as $H$ (Pa$^{0.6}$ when
`ENVELOPE_IN_PASCALS`), so the standardisation does not depend on the envelope's unit.

The lower edge is 1 Hz. Standardising tilts the spectrum towards its lowest bins, and in a
5 s window a 0.2 Hz edge passes DC and drift. A 0.5 Hz edge was tried too: the response is
unchanged, but the traces then carry per-channel offsets and slow drift across the whole
window. All three bands use a 1 Hz transition, as the filter table reports.

The supplementary version draws the same three panels from $H$ alone, without the
per-frequency standardisation, with the same EEG normalisation, filters and lags.

In [ ]:
# ===================== figure 10 options =====================================
# (band, lags in ms, x ticks) for (a), (b) and the full-width (c). The bands must be in
# STANDARDISED_BANDS, which is what pools them with their own EEG SD.
STANDARDISED_PANELS = [
    ((1.0, 30.0), (-100.0, 500.0), [-100, 0, 100, 200, 300, 400, 500]),
    ((30.0, 250.0), (-20.0, 80.0), [-20, 0, 20, 40, 60, 80]),
    ((1.0, 250.0), (-100.0, 500.0), [-50, 0, 50, 100, 150, 200, 250, 300, 350, 400]),
]
STANDARDISED_FIGSIZE = (6.27, 5.0)
STANDARDISED_HEADROOM = 1.1
# The black channel-mean envelope sits under the butterfly, in the same units, with its own zero
# moved down: its top clears the butterfly's bottom by this fraction of its own height.
STANDARDISED_ENVELOPE_GAP = 0.05
STANDARDISED_KEY_RECT = [0.0, 0.54, 0.1, 0.44]     # sensor key, in (c)'s empty pre-stimulus corner
# =============================================================================

assert all(band in STANDARDISED_BANDS for band, _, _ in STANDARDISED_PANELS)


from matplotlib.ticker import MaxNLocator


def standardised_trf(band, standardise):
    """The band's fit (its own EEG SD divided out), filtered and inverse-transformed, with or
    without the per-frequency standardisation sqrt(S_xx)."""
    spectrum = BAND_FITS[band] * BAND_RESPONSES[band][:, None]
    if standardise:
        spectrum = spectrum * np.sqrt(STIMULUS_PSD)[:, None]
    return np.roll(np.fft.irfft(spectrum, n=nperseg, axis=0), nperseg // 2, axis=0)


def draw_standardised(standardise, stem):
    fig = plt.figure(figsize=STANDARDISED_FIGSIZE)
    grid = fig.add_gridspec(2, 2, hspace=0.55, wspace=0.28)
    axes = [fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1]),
            fig.add_subplot(grid[1, :])]
    for ax, (band, lags, ticks), label in zip(axes, STANDARDISED_PANELS, ["a)", "b)", "c)"]):
        trf = standardised_trf(band, standardise)
        shown = (times >= lags[0]) & (times <= lags[1])
        for index in range(trf.shape[1]):
            ax.plot(times[shown], trf[shown, index], color=CHANNEL_COLOURS[index],
                    lw=0.5, alpha=0.55, zorder=2)
        limit = float(np.abs(trf[shown]).max()) * STANDARDISED_HEADROOM
        # Mean over channels of each channel's analytic envelope, drawn from a baseline below the
        # traces. Same units and scale as the TRFs; the hairline is the envelope's zero.
        envelope = np.abs(hilbert(trf, axis=0)).mean(-1)
        height = float(envelope[shown].max())
        baseline = -limit - height * (1 + STANDARDISED_ENVELOPE_GAP)
        ax.axhline(baseline, color=HAIRLINE, lw=0.8, zorder=0)
        ax.plot(times[shown], baseline + envelope[shown], color=INK, lw=1.2, zorder=3)
        ax.set_ylim(baseline - 0.04 * limit, limit)
        # Ticks over the TRFs' range only: below it the axis is the envelope's, offset.
        ax.set_yticks([tick for tick in MaxNLocator(5, symmetric=True).tick_values(-limit, limit)
                       if -limit <= tick <= limit])
        ax.set_xlim(*lags)
        ax.set_xticks(ticks)
        ax.axvline(0, color=HAIRLINE, lw=0.8, zorder=0)
        ax.axhline(0, color=HAIRLINE, lw=0.8, zorder=0)
        ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))
        ax.yaxis.get_offset_text().set_fontsize(SMALL_FONTSIZE)
        ax.set_title(f"{band[0]:g}-{band[1]:g} Hz", fontsize=DEFAULT_FONTSIZE)
        ax.set_xlabel("Lag (ms)")
        ax.text(-0.12 if ax is not axes[2] else -0.055, 1.02, label, transform=ax.transAxes,
                fontsize=PANEL_LABEL_FONTSIZE, fontweight="bold", va="bottom", ha="left")
        bare(ax)
    axes[0].set_ylabel("Activity (n.u.)")
    axes[2].set_ylabel("Activity (n.u.)")
    plot_sensor_key(axes[2].inset_axes(STANDARDISED_KEY_RECT))
    fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
    fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
    plt.show()


# Figure 10, and its supplementary twin without the per-frequency standardisation.
draw_standardised(True, f"trf_standardised_{REFERENCE}")
draw_standardised(False, f"trf_standardised_{REFERENCE}_supplementary_unstandardised")

# How much of each panel's energy sits below 30 Hz, with and without standardising.
for band, _, _ in STANDARDISED_PANELS:
    response = BAND_RESPONSES[band] ** 2
    shares = []
    for weight in (np.ones_like(frequencies), STIMULUS_PSD):
        energy = (np.abs(BAND_FITS[band]) ** 2).mean(-1) * weight * response
        shares.append(100 * energy[frequencies < 30].sum() / energy.sum())
    print(f"{band[0]:g}-{band[1]:g} Hz: {shares[0]:5.1f}% of the energy below 30 Hz unstandardised, "
          f"{shares[1]:5.1f}% standardised")